In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np


def resolve_meps_dir() -> Path:
    """Walk up from CWD looking for data/MEPS/(excels/)? that contains h248a.xlsx.

    Works whether the kernel CWD is the repo root, Notebooks/, or Notebooks/MEPS/.
    """
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "MEPS" / "excels",
        cwd / "data" / "MEPS",
        cwd / "MEPS" / "excels",
        cwd / "MEPS",
        cwd,
        cwd.parent / "data" / "MEPS" / "excels",
        cwd.parent / "data" / "MEPS",
        cwd.parent.parent / "data" / "MEPS" / "excels",
        cwd.parent.parent / "data" / "MEPS",
        cwd.parent.parent.parent / "data" / "MEPS" / "excels",
        cwd.parent.parent.parent / "data" / "MEPS",
    ]
    for d in candidates:
        if d.exists() and any(d.glob("h248a.xlsx")):
            return d
    return candidates[0]


MEPS_DIR = resolve_meps_dir()
print(f"MEPS directory: {MEPS_DIR}")


In [ ]:
df_248 = pd.read_excel(MEPS_DIR / "h248a.xlsx", engine="calamine")

In [ ]:
pd.set_option('display.max_columns', None)
df_248.head()

In [ ]:
clean_df = df_248[["DUPERSID","DRUGIDX","LINKIDX","RXRECIDX","RXDAYSUP","RXNAME","RXBEGYRX","RXBEGMM","RXNDC","RXXP23X","RXSF23X","TC1","TC1S1"]]


clean_df.to_excel(MEPS_DIR / "h248a_clean.xlsx")
pd.set_option('display.max_columns', None)
clean_df.head()

In [ ]:
clean_df = clean_df[clean_df["RXDAYSUP"]>0]
clean_df.head()




In [ ]:
clean_df = clean_df[clean_df["RXDAYSUP"]<990]
clean_df.shape
#clean_df.head()

In [ ]:
clean_df["RXBEGYRX"].describe()

In [ ]:
new_df_invalid = clean_df[clean_df["RXBEGYRX"]<0]
new_df_invalid.shape





In [ ]:
clean_df = clean_df[clean_df["RXBEGYRX"]>0]
clean_df.shape

In [ ]:
unique_rxname_df = (
    clean_df[["RXNAME"]]
    .drop_duplicates()
    .sort_values("RXNAME")
    .reset_index(drop=True)
)

unique_rxname_path = MEPS_DIR / "unique_rxname.csv"
unique_rxname_df.to_csv(unique_rxname_path, index=False)

print(f"Saved {len(unique_rxname_df)} unique drug names to {unique_rxname_path}")
unique_rxname_df.head()

In [ ]:
clean_df.columns.tolist()

In [ ]:
grouped_clean_df = clean_df.groupby(["DUPERSID","DRUGIDX"]).agg(
    RXDAYSUP = ("RXDAYSUP", "sum"),
    RXXP23X = ("RXXP23X", "mean"),
    RXSF23X = ("RXSF23X", "mean"),
    RXNAME = ("RXNAME", "first"),
    RXNDC = ("RXNDC", "first"),
    RXBEGYRX = ("RXBEGYRX", "first"),
    RXRECIDX = ("RXRECIDX", "first"),
    TC1 = ("TC1", "first"),
    TC1S1 = ("TC1S1", "first"),
    LINKIDX = ("LINKIDX", "first")

   
)

grouped_clean_df.head()

In [ ]:
grouped_clean_df["total_days_supply"] = np.where(grouped_clean_df["RXBEGYRX"] <=2023, 365,0).astype(int)

grouped_clean_df.shape

grouped_clean_df["total_days_supply"].describe()
#grouped_clean_df.to_excel(MEPS_DIR / "h248a_grouped.xlsx")


In [ ]:
grouped_clean_df = grouped_clean_df[grouped_clean_df["total_days_supply"]>0]


In [ ]:
grouped_clean_df["meps_adherence_ratio"] = grouped_clean_df["RXDAYSUP"]/grouped_clean_df["total_days_supply"] *100
grouped_clean_df.head()

In [ ]:
grouped_clean_df["RXDAYSUP"].describe()
grouped_clean_df.to_excel(MEPS_DIR / "h248a_Ratio.xlsx")
#grouped_clean_df = pd.read_excel(MEPS_DIR / "h248a_Ratio.xlsx", engine="calamine")


In [ ]:
grouped_clean_df.head()

In [ ]:
by_drug_class = (
    grouped_clean_df.groupby(["TC1", "TC1S1"], as_index=False)
    .agg(meps_adherence_ratio=("meps_adherence_ratio", "mean"))
    .sort_values("meps_adherence_ratio", ascending=False)
)

top_10_highest = by_drug_class.head(10)
top_10_lowest = by_drug_class.tail(10).sort_values("meps_adherence_ratio", ascending=True)

print("Top 10 highest mean adherence:")
display(top_10_highest)

print("Top 10 lowest mean adherence:")
display(top_10_lowest)

In [ ]:
top_10_highest.to_excel(MEPS_DIR / "h248a_high.xlsx")
top_10_lowest.to_excel(MEPS_DIR / "h248a_low.xlsx")

In [ ]:
chronic_df = pd.read_excel(MEPS_DIR / "is_chronic.xlsx", engine="calamine")
chronic_df.head()

In [ ]:
connecting_df = pd.read_excel(MEPS_DIR / "h248if1.xlsx", engine="calamine")
connecting_df.head()


In [ ]:
condition_df = pd.read_excel(MEPS_DIR / "h249.xlsx", engine="calamine")
condition_df.head()

In [ ]:
condition_df = condition_df[["DUPERSID","CONDIDX","ICD10CDX","AGEDIAG"]]
condition_df.head()

In [ ]:
condition_df = pd.merge(condition_df,chronic_df, on="ICD10CDX", how="left")
condition_df.head()

In [ ]:
condition_df = condition_df.drop(columns=["UNWEIGHTED","WEIGHTED_BY_perwt23f"])
condition_df.head()

In [ ]:
condition_df = condition_df[condition_df["is_chronic"]>0.0]
condition_df.head()

In [ ]:
condition_df = pd.merge(connecting_df, condition_df, on="CONDIDX", how="left")
condition_df.head()

In [ ]:
########
condition_df = condition_df.drop(columns=["DUPERSID_y","PANEL"])
condition_df = condition_df.rename(columns={"DUPERSID_x":"DUPERSID"})
condition_df = condition_df[condition_df["EVENTYPE"]==8]
condition_df.head()

In [ ]:
merged_df = pd.merge(clean_df, condition_df, left_on="LINKIDX", right_on="EVNTIDX", how="left")
merged_df.head()

In [ ]:
merged_df.dropna(subset = ["is_chronic"],inplace=True)
merged_df.head()

In [ ]:
merged_df.drop(columns=["DUPERSID_y"],inplace=True)
merged_df.rename(columns={"DUPERSID_x":"DUPERSID"},inplace=True)
merged_df = merged_df[merged_df["RXBEGYRX"]>0]
merged_df = merged_df[merged_df["RXBEGYRX"]<=2023]
merged_df = merged_df[merged_df["RXDAYSUP"]>0]
merged_df = merged_df[merged_df["RXDAYSUP"]<990]

merged_df.head()



In [ ]:
merged_df.to_excel(MEPS_DIR / "patient_conditions_df.xlsx")

In [ ]:
merged_df.columns.to_list()


In [ ]:
# --- Dedup fills before summing days (CLNK multi-condition fix) ---
# A fill (RXRECIDX) linked to N chronic conditions became N rows after
# the CLNK-condition merge. Summing RXDAYSUP on that inflated days N-fold
# and dropped every ICD but the first. Sum on the deduped fills instead;
# condition context is preserved via n_chronic_conditions,
# chronic_conditions (comma-joined list) and the bridge frame below.
fills = merged_df.drop_duplicates(subset=["DUPERSID","DRUGIDX","RXRECIDX"]).copy()
# Valid-month column for the drug-start denominator: keep 1..12, mask sentinels
fills["_valid_month"] = fills["RXBEGMM"].where(fills["RXBEGMM"].between(1, 12))

grouped_merge_df = fills.groupby(["DUPERSID","DRUGIDX"]).agg(
    RXDAYSUP = ("RXDAYSUP", "sum"),
    RXXP23X = ("RXXP23X", "mean"),
    RXSF23X = ("RXSF23X", "mean"),
    RXNAME = ("RXNAME", "first"),
    RXNDC = ("RXNDC", "first"),
    RXBEGYRX = ("RXBEGYRX", "min"),
    first_month = ("_valid_month", "min"),
    TC1 = ("TC1", "first"),
    TC1S1 = ("TC1S1", "first"),
    primary_ICD10CDX = ("ICD10CDX", "first"),
    primary_ICD10CDX_LABEL = ("ICD10CDX_LABEL", "first"),
).reset_index()

# Multi-condition context from the un-deduped merged_df
cond_context = (
    merged_df.dropna(subset=["ICD10CDX"])
    .groupby(["DUPERSID","DRUGIDX"])
    .agg(
        n_chronic_conditions=("ICD10CDX", "nunique"),
        chronic_conditions=("ICD10CDX", lambda s: ",".join(sorted(s.astype(str).unique()))),
    )
    .reset_index()
)
grouped_merge_df = grouped_merge_df.merge(cond_context, on=["DUPERSID","DRUGIDX"], how="left")
grouped_merge_df["n_chronic_conditions"] = grouped_merge_df["n_chronic_conditions"].fillna(0).astype(int)
grouped_merge_df["chronic_conditions"] = grouped_merge_df["chronic_conditions"].fillna("")

# Backwards-compat aliases so downstream cells reading ICD10CDX keep working
grouped_merge_df["ICD10CDX"] = grouped_merge_df["primary_ICD10CDX"]
grouped_merge_df["ICD10CDX_LABEL"] = grouped_merge_df["primary_ICD10CDX_LABEL"]

# Patient-drug x condition bridge for correct condition-level rollups.
# No days-supply columns -- must not be summable by accident.
patient_drug_condition = (
    merged_df.dropna(subset=["ICD10CDX"])[
        ["DUPERSID","DRUGIDX","ICD10CDX","ICD10CDX_LABEL","is_chronic","CONDIDX"]
    ]
    .drop_duplicates(subset=["DUPERSID","DRUGIDX","ICD10CDX","CONDIDX"])
    .reset_index(drop=True)
)

print(f"person-drugs: {len(grouped_merge_df):,}  |  bridge rows: {len(patient_drug_condition):,}")
grouped_merge_df.head()


In [ ]:
grouped_merge_df["total_valid_days"] = np.minimum(grouped_merge_df["RXDAYSUP"], 365).astype(int)

In [ ]:
# Drug-start-days denominator: days from earliest fill's month to Dec 31.
# Falls back to 365 when the drug started before this year OR when the
# start-month is a MEPS sentinel (-1/-7/-8/-15). Assumption: missing
# month means patient's been on it a long time and doesn't remember.
import datetime as _dt
_YEAR = 2023
_year_end = _dt.date(_YEAR, 12, 31)
_days_by_month = {m: (_year_end - _dt.date(_YEAR, m, 1)).days + 1 for m in range(1, 13)}

def _drug_start_days(first_year, first_month):
    if pd.isna(first_year) or first_year != _YEAR:
        return 365
    if pd.isna(first_month) or not (1 <= int(first_month) <= 12):
        return 365
    return _days_by_month[int(first_month)]

grouped_merge_df["drug_start_days"] = grouped_merge_df.apply(
    lambda r: _drug_start_days(r["RXBEGYRX"], r["first_month"]), axis=1
).astype(int)

grouped_merge_df["total_days_supply"] = np.where(
    grouped_merge_df["RXBEGYRX"] <= 2023,
    grouped_merge_df["drug_start_days"], 0
).astype(int)


In [ ]:
grouped_merge_df["total_valid_days"] = np.minimum(
    grouped_merge_df["total_valid_days"],
    grouped_merge_df["total_days_supply"],
)

grouped_merge_df["meps_adherence_ratio"] = np.where(
    grouped_merge_df["total_days_supply"].eq(0),
    np.nan,
    grouped_merge_df["total_valid_days"] / grouped_merge_df["total_days_supply"] * 100,
)
grouped_merge_df.head()


In [ ]:
grouped_merge_df["meps_adherence_ratio"].describe()

In [ ]:
patient_adherence = (
    grouped_merge_df.reset_index()
    .query("total_days_supply > 0")
    .groupby("DUPERSID", as_index=False)
    .agg(
        total_valid_days=("total_valid_days", "sum"),
        total_days_supply=("total_days_supply", "sum"),
        drug_count=("DRUGIDX", "count"),
    )
)
patient_adherence["meps_adherence_ratio"] = (
    patient_adherence["total_valid_days"] / patient_adherence["total_days_supply"] * 100
)

top_10_high = patient_adherence.nlargest(10, "meps_adherence_ratio")
top_10_low = patient_adherence.nsmallest(10, "meps_adherence_ratio")

output_dir = str(MEPS_DIR)
top_10_high.to_excel(f"{output_dir}/top_10_high.xlsx", index=False)
top_10_low.to_excel(f"{output_dir}/top_10_low.xlsx", index=False)

print("Top 10 patients with highest adherence:")
display(top_10_high)

print("Top 10 patients with lowest adherence:")
display(top_10_low)

In [ ]:
patient_drug_df = (
    grouped_merge_df.reset_index()
    .query("total_days_supply > 0")
)

top_10_high = patient_drug_df.nlargest(10, "meps_adherence_ratio")
top_10_low = patient_drug_df.nsmallest(10, "meps_adherence_ratio")

output_dir = str(MEPS_DIR)
top_10_high.to_excel(f"{output_dir}/top_10_high.xlsx", index=False)
top_10_low.to_excel(f"{output_dir}/top_10_low.xlsx", index=False)

print("Top 10 patient-drug pairs with highest adherence:")
display(top_10_high)

print("Top 10 patient-drug pairs with lowest adherence:")
display(top_10_low)

In [ ]:
adherence_threshold = 60


In [ ]:
print(grouped_merge_df["ICD10CDX"].nunique())
print(grouped_merge_df["TC1"].nunique())
print(grouped_merge_df["RXNAME"].nunique())
print(grouped_merge_df["TC1S1"].nunique())

In [ ]:
grouped_merge_df.to_excel(MEPS_DIR / "grouped_merge_df.xlsx")

In [ ]:
import matplotlib.pyplot as plt

by_condition = (
    grouped_merge_df.reset_index()
    .groupby(["ICD10CDX", "ICD10CDX_LABEL"], as_index=False)
    .agg(meps_adherence_ratio=("meps_adherence_ratio", "mean"))
)

adherence_threshold = 60
bins = np.arange(0, 110, 10)

fig, ax = plt.subplots(figsize=(12, 6))
counts, edges, patches = ax.hist(
    by_condition["meps_adherence_ratio"],
    bins=bins,
    edgecolor="black",
    linewidth=0.6,
)

for patch, left, right in zip(patches, edges[:-1], edges[1:]):
    if right <= adherence_threshold:
        patch.set_facecolor("#d9534f")
    elif left >= adherence_threshold:
        patch.set_facecolor("#5cb85c")
    else:
        patch.set_facecolor("#f0ad4e")

ax.axvline(
    adherence_threshold,
    color="red",
    linestyle=":",
    linewidth=2,
    label=f"{adherence_threshold}% adherence threshold",
)

bin_labels = [f"{int(left)}-{int(right)}" for left, right in zip(edges[:-1], edges[1:])]
ax.set_xticks(edges[:-1] + 5)
ax.set_xticklabels(bin_labels, rotation=45, ha="right")

n_adherent = (by_condition["meps_adherence_ratio"] >= adherence_threshold).sum()
n_total = len(by_condition)

ax.set_xlabel("Average MEPS adherence ratio by condition (%)")
ax.set_ylabel("Number of conditions (ICD10CDX)")
ax.set_title(
    f"Distribution of condition-level average adherence\n"
    f"{n_adherent} of {n_total} conditions meet the {adherence_threshold}% threshold"
)
ax.legend()
plt.tight_layout()
plt.show()

adherent_conditions = by_condition[
    by_condition["meps_adherence_ratio"] >= adherence_threshold
].sort_values("meps_adherence_ratio", ascending=False)

print(f"Adherent conditions (>={adherence_threshold}%):")
display(adherent_conditions)

In [ ]:
from sklearn.preprocessing import LabelEncoder

corr_df = grouped_merge_df.reset_index()[
    ["ICD10CDX", "TC1", "TC1S1", "meps_adherence_ratio"]
].dropna()

corr_df["ICD10CDX_code"] = LabelEncoder().fit_transform(corr_df["ICD10CDX"].astype(str))


def eta_squared(categorical, continuous):
    """Share of adherence variance explained by a categorical variable."""
    categories = categorical.astype(str)
    grand_mean = continuous.mean()
    ss_total = ((continuous - grand_mean) ** 2).sum()
    ss_between = sum(
        len(continuous[categories == cat]) * (continuous[categories == cat].mean() - grand_mean) ** 2
        for cat in categories.unique()
    )
    return ss_between / ss_total if ss_total > 0 else np.nan


pearson_corr = corr_df[
    ["ICD10CDX_code", "TC1", "TC1S1", "meps_adherence_ratio"]
].corr(method="pearson")

pearson_with_adherence = pearson_corr["meps_adherence_ratio"].drop("meps_adherence_ratio")

eta_results = pd.Series(
    {
        "ICD10CDX": eta_squared(corr_df["ICD10CDX"], corr_df["meps_adherence_ratio"]),
        "TC1": eta_squared(corr_df["TC1"], corr_df["meps_adherence_ratio"]),
        "TC1S1": eta_squared(corr_df["TC1S1"], corr_df["meps_adherence_ratio"]),
    },
    name="eta_squared",
)

X = pd.get_dummies(
    corr_df[["ICD10CDX", "TC1", "TC1S1"]].astype(str),
    drop_first=True,
).astype(float)
y = corr_df["meps_adherence_ratio"].values
X_with_intercept = np.column_stack([np.ones(len(X)), X.values])
coeffs, _, _, _ = np.linalg.lstsq(X_with_intercept, y, rcond=None)
y_pred = X_with_intercept @ coeffs
ss_res = ((y - y_pred) ** 2).sum()
ss_tot = ((y - y.mean()) ** 2).sum()
combined_r2 = 1 - ss_res / ss_tot
n, p = len(y), X.shape[1]
adj_r2 = 1 - (1 - combined_r2) * (n - 1) / (n - p - 1)

print("Pearson correlation with meps_adherence_ratio")
print("(ICD10CDX uses label encoding; interpret as a weak linear proxy only)\n")
display(pearson_with_adherence.to_frame("pearson_r"))

print("\nEta-squared with meps_adherence_ratio")
print("(better measure for categorical variables: share of variance explained)\n")
display(eta_results.to_frame())

print("\nCombined model: ICD10CDX + TC1 + TC1S1 together")
print(f"  R-squared: {combined_r2:.4f}")
print(f"  Adjusted R-squared: {adj_r2:.4f}")

heatmap_data = pearson_corr.loc[
    ["ICD10CDX_code", "TC1", "TC1S1", "meps_adherence_ratio"],
    ["ICD10CDX_code", "TC1", "TC1S1", "meps_adherence_ratio"],
].rename(index={"ICD10CDX_code": "ICD10CDX"}, columns={"ICD10CDX_code": "ICD10CDX"})

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(heatmap_data.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(heatmap_data.columns)), heatmap_data.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_data.index)), heatmap_data.index)
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        ax.text(j, i, f"{heatmap_data.iloc[i, j]:.2f}", ha="center", va="center", color="black")
fig.colorbar(im, ax=ax)
ax.set_title("Pearson correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
by_tc1s1 = (
    grouped_merge_df.reset_index()
    .groupby(["TC1", "TC1S1"], as_index=False)
    .agg(meps_adherence_ratio=("meps_adherence_ratio", "mean"))
)

adherence_threshold = 60
bins = np.arange(0, 110, 10)

fig, ax = plt.subplots(figsize=(12, 6))
counts, edges, patches = ax.hist(
    by_tc1s1["meps_adherence_ratio"],
    bins=bins,
    edgecolor="black",
    linewidth=0.6,
)

for patch, left, right in zip(patches, edges[:-1], edges[1:]):
    if right <= adherence_threshold:
        patch.set_facecolor("#d9534f")
    elif left >= adherence_threshold:
        patch.set_facecolor("#5cb85c")
    else:
        patch.set_facecolor("#f0ad4e")

ax.axvline(
    adherence_threshold,
    color="red",
    linestyle=":",
    linewidth=2,
    label=f"{adherence_threshold}% adherence threshold",
)

bin_labels = [f"{int(left)}-{int(right)}" for left, right in zip(edges[:-1], edges[1:])]
ax.set_xticks(edges[:-1] + 5)
ax.set_xticklabels(bin_labels, rotation=45, ha="right")

n_adherent = (by_tc1s1["meps_adherence_ratio"] >= adherence_threshold).sum()
n_total = len(by_tc1s1)

ax.set_xlabel("Average MEPS adherence ratio by TC1S1 drug subclass (%)")
ax.set_ylabel("Number of TC1S1 drug subclasses")
ax.set_title(
    f"Distribution of TC1S1-level average adherence\n"
    f"{n_adherent} of {n_total} drug subclasses meet the {adherence_threshold}% threshold"
)
ax.legend()
plt.tight_layout()
plt.show()

adherent_tc1s1 = by_tc1s1[
    by_tc1s1["meps_adherence_ratio"] >= adherence_threshold
].sort_values("meps_adherence_ratio", ascending=False)

print(f"Adherent TC1S1 drug subclasses (>={adherence_threshold}%):")
display(adherent_tc1s1)

In [ ]:
low_adherence_threshold = 10

by_tc1s1 = (
    grouped_merge_df.reset_index()
    .groupby(["TC1", "TC1S1"], as_index=False)
    .agg(meps_adherence_ratio=("meps_adherence_ratio", "mean"))
)

by_condition = (
    grouped_merge_df.reset_index()
    .groupby(["ICD10CDX", "ICD10CDX_LABEL"], as_index=False)
    .agg(meps_adherence_ratio=("meps_adherence_ratio", "mean"))
)

low_tc1s1 = (
    by_tc1s1[by_tc1s1["meps_adherence_ratio"] < low_adherence_threshold]
    .sort_values("meps_adherence_ratio", ascending=True)
)

low_conditions = (
    by_condition[by_condition["meps_adherence_ratio"] < low_adherence_threshold]
    .sort_values("meps_adherence_ratio", ascending=True)
)

print(f"TC1S1 drug subclasses with average adherence < {low_adherence_threshold}%:")
print(f"Count: {len(low_tc1s1)} of {len(by_tc1s1)}")
display(low_tc1s1)

print(f"\nICD10CDX conditions with average adherence < {low_adherence_threshold}%:")
print(f"Count: {len(low_conditions)} of {len(by_condition)}")
display(low_conditions)

In [ ]:
demo_df = pd.read_excel(MEPS_DIR / "h251.xlsx", engine="calamine")
demo_df.head()

In [ ]:
new_demo_df = demo_df[["DUPERSID","AGE23X","SEX","INSCOV23","POVCAT23","FAMINC23","PSTATS31","PSTATS53",
"PSTATS42","RACEV2X","BEGRFM31", "BEGRFY31", "BEGRFM42", "BEGRFY42", "BEGRFM53","BEGRFY53", "ENDRFM31", "ENDRFY31",
 "ENDRFM42", "ENDRFY42", "ENDRFM53",
"ENDRFY53","DLAYPM42","DLAYCA42"]]
new_demo_df.head()

In [ ]:
##1 - Any private (Person had any private insurance coverage [including
#TRICARE/CHAMPVA] at any time during 2023)
#2 - Public only (Person had only public insurance coverage [excluding
#TRICARE/CHAMPVA] during 2023)
#3 - Uninsured (Person was uninsured during all of 2023)

In [ ]:
#POVCAT23 - Poverty level

#1 is Poor
#5 is high income

#PSTATS31 - if val is 31 - deseaced 

In [ ]:

new_grouped_merge_df = grouped_merge_df.merge(new_demo_df, on="DUPERSID", how="left")
new_grouped_merge_df.head()


In [ ]:
pstat_cols = ["PSTATS31", "PSTATS53", "PSTATS42"]
missing = [c for c in pstat_cols if c not in new_grouped_merge_df.columns]
if missing:
    raise KeyError(
        f"Missing {missing}. Re-run from the new_demo_df / merge cells "
        "(PSTATS columns are dropped later after reference-days are computed)."
    )

pstat_values = [31, 23, 24, 61]

pstat_df = new_grouped_merge_df.loc[
    new_grouped_merge_df["PSTATS31"].isin(pstat_values)
    | new_grouped_merge_df["PSTATS53"].isin(pstat_values)
    | new_grouped_merge_df["PSTATS42"].isin(pstat_values)
]

print(pstat_df.shape)
pstat_df.head()

In [ ]:
pstat_values = [31,11, 12, 13, 14, 21, 22, 31, 32, 33, 34, 35, 41, 42, 43, 44, 51, 74, -1  ]
pstat_cols = ["PSTATS31", "PSTATS53", "PSTATS42"]
missing = [c for c in pstat_cols if c not in new_grouped_merge_df.columns]
if missing:
    raise KeyError(
        f"Missing {missing}. Re-run from the new_demo_df / merge cells "
        "(PSTATS columns are dropped later after reference-days are computed)."
    )

for val in pstat_values:
    mask = new_grouped_merge_df[pstat_cols].eq(val).any(axis=1)
    subset = new_grouped_merge_df.loc[mask, "total_valid_days"]
    print(f"PSTATS value {val} (rows where any PSTATS column == {val}): {len(subset)}")
    if len(subset) == 0:
        print("  No rows found.\n")
    else:
        display(subset.describe().to_frame(f"total_valid_days (PSTATS={val})"))
        print()

In [ ]:
from datetime import date
import calendar

# Year this notebook targets. Change if you clone this notebook to a new year.
YEAR = 2023

# PSTATS codes per h251doc.pdf Tables 7 and 8 (pp. C-14 to C-20).
# The MEPS PSTATS taxonomy is stable across years; the same codes apply
# to 2020-2023 (COVID year 2020 just has more -1 nonresponse).
PSTATS_LABELS = {
    11: "In RU, responded in person (full round)",
    12: "Active military duty (no survey)",
    13: "Moved from RU, known whereabouts",
    14: "Moved from RU, unknown whereabouts",
    21: "Hospitalized (rounds 4-5)",
    22: "Left institution, rejoined community (rounds 4-5)",
    23: "Left institution then died (rounds 4-5)",
    24: "Died in health-care institution (rounds 4-5)",
    31: "Deceased",
    32: "Institutionalized (health care)",
    33: "Institutionalized (non-health care)",
    34: "Moved outside US",
    35: "Moved to military facility on active duty",
    36: "Went to institution (type unknown)",
    41: "Joined RU after reference period began",
    42: "Joined RU, not full-time military",
    43: "Unknown disposition / moved to unknown location",
    44: "Moved to another RU in same PSU",
    51: "Newborn in reference period",
    62: "Institutionalized prior to reference period (R3/1 only)",
    71: "Non-Key student living away, grades 1-12",
    72: "Dropped as ineligible non-Key",
    73: "Non-Key moved without Key member",
    74: "Moved as FT military, no Key member (ineligible round)",
    81: "FT student living away, non-response",
    -1: "Not in this round",
}

# Full-round participants: use the whole reference-period window as-is.
FULL_ROUND_STATUSES = {11, 13, 14, 22, 41, 42, 44, 51, 71}
# Ended-early participants (excluding death, handled separately below):
# reference period cuts off partway through the round.
STOP_STATUSES = {32, 33, 34, 35, 36}
# No coverage in this round (Inapplicable reference period, ineligible,
# or out-of-scope for the whole round).
NO_COVERAGE_STATUSES = {0, 12, 21, 24, 43, 62, 63, 64, 72, 73, 74, 81}


def _month_start(year, month):
    if pd.isna(month) or pd.isna(year) or month <= 0 or year <= 0:
        return None
    return date(int(year), int(month), 1)


def _month_end(year, month):
    if pd.isna(month) or pd.isna(year) or month <= 0 or year <= 0:
        return None
    return date(int(year), int(month), calendar.monthrange(int(year), int(month))[1])


def _round_detail(row, round_suffix):
    ps = int(row[f"PSTATS{round_suffix}"])
    return {
        "round": round_suffix,
        "pstats": ps,
        "pstats_label": PSTATS_LABELS.get(ps, f"Code {ps}"),
        "ref_start": _month_start(row[f"BEGRFY{round_suffix}"], row[f"BEGRFM{round_suffix}"]),
        "ref_end": _month_end(row[f"ENDRFY{round_suffix}"], row[f"ENDRFM{round_suffix}"]),
    }


def compute_reference_coverage(row, year=YEAR):
    """Return eligible days, coverage start/end, and notes from PSTATS + BEGRF/ENDRF.

    PSTATS == -1 means the person was not in that round (skip; no days added or removed).
    """
    year_start = date(year, 1, 1)
    year_end = date(year, 12, 31)

    active_rounds = [
        sfx for sfx in [31, 42, 53] if int(row[f"PSTATS{sfx}"]) != -1
    ]
    if not active_rounds:
        return 0, None, None, "not_in_any_round"

    if all(int(row[f"PSTATS{sfx}"]) == 11 for sfx in [31, 42, 53]):
        return 365, year_start, year_end, "full_year_all_rounds_11"

    coverage_start = None
    coverage_end = None
    notes = []

    for round_suffix in [31, 42, 53]:
        ps = int(row[f"PSTATS{round_suffix}"])
        beg = _month_start(row[f"BEGRFY{round_suffix}"], row[f"BEGRFM{round_suffix}"])
        end = _month_end(row[f"ENDRFY{round_suffix}"], row[f"ENDRFM{round_suffix}"])

        if ps == -1:
            notes.append(f"R{round_suffix}:not_in_round")
            continue
        if ps in NO_COVERAGE_STATUSES:
            notes.append(f"R{round_suffix}:no_coverage({ps})")
            continue

        if coverage_start is None:
            coverage_start = max(beg, year_start) if beg else year_start

        if ps in FULL_ROUND_STATUSES:
            if end:
                coverage_end = min(end, year_end)
            notes.append(f"R{round_suffix}:active({ps})")
        elif ps == 31:
            if end:
                coverage_end = min(end, year_end)
            notes.append(f"R{round_suffix}:death")
            break
        elif ps in STOP_STATUSES:
            if end:
                coverage_end = min(end, year_end)
            notes.append(f"R{round_suffix}:stop({ps})")
            break
        else:
            # Unknown PSTATS code — leave a marker instead of silently including.
            notes.append(f"R{round_suffix}:UNCLASSIFIED({ps})")

    if coverage_start is None or coverage_end is None or coverage_end < coverage_start:
        return 0, None, None, "no_coverage"

    days = (coverage_end - coverage_start).days + 1
    return days, coverage_start, coverage_end, ";".join(notes)


person_demo = new_demo_df.drop_duplicates("DUPERSID").copy()
coverage_parts = person_demo.apply(compute_reference_coverage, axis=1, result_type="expand")
person_demo["total_days_supply"] = coverage_parts[0]
person_demo["ref_start_2023"] = coverage_parts[1]
person_demo["ref_end_2023"] = coverage_parts[2]
person_demo["coverage_notes"] = coverage_parts[3]

person_demo["in_round_31"] = person_demo["PSTATS31"] != -1
person_demo["in_round_42"] = person_demo["PSTATS42"] != -1
person_demo["in_round_53"] = person_demo["PSTATS53"] != -1

# Flag: person missed the last round (R5/3). Their denominator is truncated
# at the R4/2 interview date, but any fill captured at R4/2 with a long
# RXDAYSUP will keep adding days past that endpoint. Result: MPR can be
# biased upward (often pinned to 100% by the np.minimum cap below).
person_demo["r53_nonresponse"] = person_demo["PSTATS53"] == -1

person_demo["participation_type"] = np.select(
    [
        person_demo["coverage_notes"].eq("full_year_all_rounds_11"),
        person_demo["coverage_notes"].eq("not_in_any_round"),
        person_demo["coverage_notes"].str.contains("death", na=False),
        person_demo["coverage_notes"].str.contains("stop", na=False),
        person_demo["coverage_notes"].str.contains("no_coverage", na=False),
        person_demo["total_days_supply"].eq(0),
    ],
    [
        "full_year",
        "not_in_any_round",
        "ended_early_death",
        "ended_early_left_ru",
        "partial_no_survey",
        "no_coverage",
    ],
    default="partial_year",
)

reference_days_df = person_demo[
    [
        "DUPERSID",
        "in_round_31",
        "in_round_42",
        "in_round_53",
        "ref_start_2023",
        "ref_end_2023",
        "total_days_supply",
        "participation_type",
        "coverage_notes",
        "r53_nonresponse",
    ]
]

ROUND_DETAIL_COLS = [
    "PSTATS31", "PSTATS42", "PSTATS53",
    "BEGRFM31", "BEGRFY31", "ENDRFM31", "ENDRFY31",
    "BEGRFM42", "BEGRFY42", "ENDRFM42", "ENDRFY42",
    "BEGRFM53", "BEGRFY53", "ENDRFM53", "ENDRFY53",
]
new_demo_df = new_demo_df.drop(columns=ROUND_DETAIL_COLS, errors="ignore")

print("Participation type counts:")
display(reference_days_df["participation_type"].value_counts().to_frame("persons"))

# Diagnostic: how many persons might have MPR biased upward by R5/3 nonresponse?
n_r53 = int(reference_days_df["r53_nonresponse"].sum())
print(f"\nR5/3 nonresponders flagged (PSTATS53 == -1): {n_r53} persons")
print("  Their denominator cuts off at the R4/2 interview date, but their")
print("  numerator (sum RXDAYSUP) can include fill days that extend past it.")
print("  Downstream: their MPR is at risk of being biased HIGH (often 100%).")

# Diagnostic: were any PSTATS codes in the data left UNCLASSIFIED after
# the extended classification above?
unclassified = person_demo[person_demo["coverage_notes"].str.contains("UNCLASSIFIED", na=False)]
if len(unclassified):
    print(f"\nWARNING: {len(unclassified)} persons still have unclassified PSTATS values.")
    print("  Inspect person_demo[person_demo['coverage_notes'].str.contains('UNCLASSIFIED')].")
else:
    print("\nAll PSTATS codes in this year's data were classified. ✓")

print("\nSample: full-year vs partial vs deceased")
display(
    reference_days_df[
        reference_days_df["participation_type"].isin(
            ["full_year", "ended_early_death", "partial_year", "partial_no_survey"]
        )
    ].head(10)
)


In [ ]:
new_grouped_merge_df = new_grouped_merge_df.drop(
    columns=[
        "total_days_supply",
        "ref_start_2023",
        "ref_end_2023",
        "participation_type",
        "coverage_notes",
        "r53_nonresponse",
    ],
    errors="ignore",
)

new_grouped_merge_df = new_grouped_merge_df.merge(
    reference_days_df[
        [
            "DUPERSID",
            "total_days_supply",
            "ref_start_2023",
            "ref_end_2023",
            "participation_type",
            "coverage_notes",
            "r53_nonresponse",
        ]
    ],
    on="DUPERSID",
    how="left",
)

new_grouped_merge_df["total_valid_days"] = np.minimum(
    new_grouped_merge_df["total_valid_days"],
    new_grouped_merge_df["total_days_supply"],
)

new_grouped_merge_df["meps_adherence_ratio"] = np.where(
    new_grouped_merge_df["total_days_supply"].eq(0),
    np.nan,
    new_grouped_merge_df["total_valid_days"] / new_grouped_merge_df["total_days_supply"] * 100,
)

ROUND_DETAIL_COLS = [
    "PSTATS31", "PSTATS42", "PSTATS53",
    "BEGRFM31", "BEGRFY31", "ENDRFM31", "ENDRFY31",
    "BEGRFM42", "BEGRFY42", "ENDRFM42", "ENDRFY42",
    "BEGRFM53", "BEGRFY53", "ENDRFM53", "ENDRFY53",
]
new_grouped_merge_df = new_grouped_merge_df.drop(columns=ROUND_DETAIL_COLS, errors="ignore")

print("total_days_supply (PSTATS-based) summary:")
display(new_grouped_merge_df["total_days_supply"].describe().to_frame())

print("\nmeps_adherence_ratio after update:")
display(new_grouped_merge_df["meps_adherence_ratio"].describe().to_frame())

# Diagnostic: how much does R5/3-nonresponse bias inflate mean MPR?
overall = new_grouped_merge_df["meps_adherence_ratio"]
excl_r53 = new_grouped_merge_df.loc[
    ~new_grouped_merge_df["r53_nonresponse"].fillna(False), "meps_adherence_ratio"
]
n_r53_pairs = int(new_grouped_merge_df["r53_nonresponse"].fillna(False).sum())
print(f"\nR5/3-nonresponse impact on person-drug pairs: {n_r53_pairs} pairs affected")
print(f"  mean MPR overall:                    {overall.mean():.2f}%")
print(f"  mean MPR excluding R5/3 nonresponders: {excl_r53.mean():.2f}%")
print("  (If the excluded mean is materially lower, the flagged rows are")
print("   inflating the ratio via numerator-bleed past the truncated denominator.)")

print("\nCompare old flat 365 vs new PSTATS-based denominator:")
compare = reference_days_df.groupby("participation_type").agg(
    persons=("DUPERSID", "count"),
    mean_days=("total_days_supply", "mean"),
    min_days=("total_days_supply", "min"),
    max_days=("total_days_supply", "max"),
)
display(compare)


In [ ]:
new_grouped_merge_df.head()

In [ ]:
rxname_df = pd.read_csv(MEPS_DIR / "unique_rxname_chronic_labeled_revised.csv")
rxname_df.head()

In [ ]:
new_grouped_merge_df = pd.merge(new_grouped_merge_df, rxname_df, on="RXNAME", how="left")
new_grouped_merge_df.head()

In [ ]:
#new_grouped_merge_df.drop(columns = ["drug_condition_type_y","is_chronic_y"],inplace = True)

In [ ]:
new_grouped_merge_df["is_chronic"].value_counts()

In [ ]:
new_grouped_merge_df = new_grouped_merge_df[new_grouped_merge_df["is_chronic"]==1]
new_grouped_merge_df.head()

In [ ]:
by_drug_class = (
    new_grouped_merge_df.groupby(["TC1", "TC1S1"], as_index=False)
    .agg(meps_adherence_ratio=("meps_adherence_ratio", "mean"))
    .sort_values("meps_adherence_ratio", ascending=False)
)

top_10_highest = by_drug_class.head(10)
top_10_lowest = by_drug_class.tail(10).sort_values("meps_adherence_ratio", ascending=True)

print("Top 10 highest mean adherence:")
display(top_10_highest)

print("Top 10 lowest mean adherence:")
display(top_10_lowest)

In [ ]:
low_adherence_threshold = 10

by_tc1s1 = (
    new_grouped_merge_df.reset_index()
    .groupby(["TC1", "TC1S1"], as_index=False)
    .agg(meps_adherence_ratio=("meps_adherence_ratio", "mean"))
)

by_condition = (
    new_grouped_merge_df.reset_index()
    .groupby(["ICD10CDX", "ICD10CDX_LABEL"], as_index=False)
    .agg(meps_adherence_ratio=("meps_adherence_ratio", "mean"))
)

low_tc1s1 = (
    by_tc1s1[by_tc1s1["meps_adherence_ratio"] < low_adherence_threshold]
    .sort_values("meps_adherence_ratio", ascending=True)
)

low_conditions = (
    by_condition[by_condition["meps_adherence_ratio"] < low_adherence_threshold]
    .sort_values("meps_adherence_ratio", ascending=True)
)

print(f"TC1S1 drug subclasses with average adherence < {low_adherence_threshold}%:")
print(f"Count: {len(low_tc1s1)} of {len(by_tc1s1)}")
display(low_tc1s1)

print(f"\nICD10CDX conditions with average adherence < {low_adherence_threshold}%:")
print(f"Count: {len(low_conditions)} of {len(by_condition)}")
display(low_conditions)

In [ ]:
top_20_least_adherent = (
    new_grouped_merge_df
    .dropna(subset=["meps_adherence_ratio"])
    .groupby(["ICD10CDX", "ICD10CDX_LABEL", "RXNAME"], as_index=False)
    .agg(
        meps_adherence_ratio=("meps_adherence_ratio", "mean"),
        patient_drug_pairs=("DUPERSID", "count"),
    )
    .nsmallest(20, "meps_adherence_ratio")
    .sort_values("meps_adherence_ratio", ascending=True)
    .reset_index(drop=True)
)

print("Top 20 least adherent conditions with drug name:")
display(top_20_least_adherent)

In [ ]:
new_grouped_merge_df.head()

In [ ]:
new_grouped_merge_df.columns.tolist()

In [ ]:
delay_df = demo_df[["DUPERSID", "DLAYPM42", "DLAYCA42"]].drop_duplicates("DUPERSID")
new_grouped_merge_df = new_grouped_merge_df.drop(
    columns=["DLAYPM42", "DLAYCA42"], errors="ignore"
).merge(delay_df, on="DUPERSID", how="left")

_model_cols = [
    'DUPERSID','RXNAME','ICD10CDX','ICD10CDX_LABEL',
    'AGE23X','SEX','INSCOV23','POVCAT23','FAMINC23','RACEV2X',
    'RXSF23X','RXXP23X',
    'DLAYPM42','DLAYCA42','meps_adherence_ratio',
    # optional: run the medication_freq / medication_dose cell first to attach these
    'medication_freq', 'medication_dose', 'medication_dose_unit', 'medication_qty_unit',
]
model_df = new_grouped_merge_df[
    [c for c in _model_cols if c in new_grouped_merge_df.columns]
].copy()

# Cost cols: NaN -> 0, clip negatives (MEPS RXSF/RXXP should be >= 0)
model_df['RXSF23X'] = pd.to_numeric(model_df['RXSF23X'], errors='coerce').fillna(0).clip(lower=0)
model_df['RXXP23X'] = pd.to_numeric(model_df['RXXP23X'], errors='coerce').fillna(0).clip(lower=0)

model_df = model_df.rename(columns={'RXSF23X':'RXSF_PATIENT','RXXP23X':'RXXP_TOTAL'})
model_df.head()

In [ ]:
model_df.head()

In [ ]:
model_df['is_adherent'] = np.where(model_df['meps_adherence_ratio'] >= 60, 1, 0)

neg_mask = model_df.select_dtypes(include="number").lt(0).any(axis=1)
n_dropped = int(neg_mask.sum())
model_df = model_df.loc[~neg_mask].copy()
print(f"Dropped RXNDC earlier; dropped {n_dropped} rows with negatives; remaining {len(model_df)}")

model_df.head()

In [ ]:
# Drop RXNDC (if still present) and remove any row with a negative numeric value
model_df = model_df.drop(columns=["RXNDC"], errors="ignore")

feature_cols = [
    c for c in model_df.columns
    if c not in {"DUPERSID", "RXNAME", "ICD10CDX", "ICD10CDX_LABEL"}
]
for c in feature_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

print("Negative counts BEFORE drop:")
print(model_df[feature_cols].lt(0).sum().sort_values(ascending=False))

neg_mask = model_df[feature_cols].lt(0).any(axis=1)
n_before = len(model_df)
model_df = model_df.loc[~neg_mask].copy()

print(f"\nDropped {int(neg_mask.sum())} rows with negatives ({n_before} -> {len(model_df)})")
print("Negative counts AFTER drop:")
print(model_df[feature_cols].lt(0).sum().sort_values(ascending=False))
model_df.head()


In [ ]:
model_df.describe()

In [ ]:
model_df['INSCOV23'].unique()

In [ ]:
# Insurance: 3 dummies matching MEPS INSCOV23 codes (1=private, 2=public-only, 3=uninsured)
model_df['INSCOV_PRIVATE'] = (model_df['INSCOV23'] == 1).astype(int)
model_df['INSCOV_PUBLIC'] = (model_df['INSCOV23'] == 2).astype(int)
model_df['INSCOV_UNINSURED'] = (model_df['INSCOV23'] == 3).astype(int)
model_df = model_df.drop(columns=['INSCOV23'])
model_df.head()

In [ ]:
model_df['MALE'] = np.where(model_df['SEX']==1, 1, 0)
model_df['FEMALE'] = np.where(model_df['SEX']==2, 1, 0)
model_df.drop(columns = ['SEX'],inplace = True)
model_df.head()


In [ ]:
model_df['RACEV2X'].unique()


model_df['WHITE'] = np.where(model_df['RACEV2X']==1, 1, 0)
model_df['BLACK'] = np.where(model_df['RACEV2X']==2, 1, 0)
model_df['AMER_INDIAN'] = np.where(model_df['RACEV2X']==3, 1, 0)
model_df['ASIAN_INDIAN'] = np.where(model_df['RACEV2X']==4, 1, 0)
model_df['CHINESE'] = np.where(model_df['RACEV2X']==5, 1, 0)
model_df['FILIPINO'] = np.where(model_df['RACEV2X']==6, 1, 0)


# 1 WHITE - NO OTHER RACE REPORTED 14,118 250,268,262
# 2 BLACK - NO OTHER RACE REPORTED 2,666 44,558,431
# 3 AMER INDIAN/ALASKA NATIVE-NO OTHER RACE 163 2,838,974
# 4 ASIAN INDIAN - NO OTHER RACE REPORTED 380 7,055,957
# 5 CHINESE - NO OTHER RACE REPORTED 272 5,400,356
# 6 FILIPINO - NO OTHER RACE REPORTED 174 3,110,110
# 10 OTH ASIAN/NATV HAWAIIAN/PACFC ISL-NO OTH 422 7,436,055
# 12 MULTIPLE RACES REPORTED 
model_df.drop(columns = ['RACEV2X'],inplace = True)
model_df.head()


In [ ]:
# DLAYPM42 / DLAYCA42 may already be dropped — pull them back from demo_df if needed
if "DLAYPM42" not in model_df.columns or "DLAYCA42" not in model_df.columns:
    delay_df = demo_df[["DUPERSID", "DLAYPM42", "DLAYCA42"]].drop_duplicates("DUPERSID")
    model_df = model_df.drop(columns=["DLAYPM42", "DLAYCA42"], errors="ignore").merge(
        delay_df, on="DUPERSID", how="left"
    )

# 1=YES delayed for cost, 2=NO
model_df["PMED_DELAY_COST"] = np.where(model_df["DLAYPM42"] == 1, 1, 0)
model_df["NO_PMED_DELAY_COST"] = np.where(model_df["DLAYPM42"] == 2, 1, 0)

model_df["CARE_DELAY_COST"] = np.where(model_df["DLAYCA42"] == 1, 1, 0)
model_df["NO_CARE_DELAY_COST"] = np.where(model_df["DLAYCA42"] == 2, 1, 0)

model_df.drop(columns=["DLAYPM42", "DLAYCA42"], inplace=True)
model_df.head()


In [ ]:
# POVCAT23 may already be dropped — pull it back from demo_df if needed
if "POVCAT23" not in model_df.columns:
    pov_df = demo_df[["DUPERSID", "POVCAT23"]].drop_duplicates("DUPERSID")
    model_df = model_df.merge(pov_df, on="DUPERSID", how="left")

# 1=Poor ... 5=High income
model_df["POV_POOR"] = np.where(model_df["POVCAT23"] == 1, 1, 0)
model_df["POV_NEAR_POOR"] = np.where(model_df["POVCAT23"] == 2, 1, 0)
model_df["POV_LOW"] = np.where(model_df["POVCAT23"] == 3, 1, 0)
model_df["POV_MIDDLE"] = np.where(model_df["POVCAT23"] == 4, 1, 0)
model_df["POV_HIGH"] = np.where(model_df["POVCAT23"] == 5, 1, 0)

model_df.drop(columns=["POVCAT23"], inplace=True)
model_df.head()


In [ ]:
model_df["ICD10CDX_LABEL"].nunique()

In [ ]:
model_df["RXNAME"].value_counts().tolist()

In [ ]:
rx_counts = model_df["RXNAME"].value_counts(dropna=False)
n_once = int((rx_counts >9).sum())

print(f"unique RXNAME: {rx_counts.shape[0]}")
print(f"RXNAME appearing exactly once: {n_once}")
print(f"share of unique names: {100 * n_once / rx_counts.shape[0]:.1f}%")
print(f"share of rows: {100 * n_once / len(model_df):.1f}%")
rx_counts[rx_counts >9 ]


In [ ]:
# Diagnostic: how many rows/RXNAMEs we WOULD have kept at a >=10 threshold
# (kept for reference — we actually keep ALL drugs; the >=10 filter was too aggressive)
rx_counts = model_df['RXNAME'].value_counts()
keep_names = rx_counts[rx_counts >= 10].index
print(f"total pair-level rows: {len(model_df):,}")
print(f"unique RXNAME: {model_df['RXNAME'].nunique():,}")
print(f"RXNAMEs with >=10 fills: {len(keep_names):,}")
print(f"rows if we HAD applied >=10 filter: {int(model_df['RXNAME'].isin(keep_names).sum()):,} "
      f"({int(model_df['RXNAME'].isin(keep_names).mean()*100)}% retained)")
print("Not filtering — cell 90 will one-hot all RXNAMEs.")

In [ ]:
model_df["ICD10CDX"].nunique()

In [ ]:
model_df["ICD10CDX"].value_counts()

In [ ]:
icd_counts = model_df["ICD10CDX"].value_counts(dropna=False)
n_once = int((icd_counts >9).sum())

print(f"unique ICD10CDX: {icd_counts.shape[0]}")
print(f"ICD10CDX appearing exactly once: {n_once}")
print(f"share of unique codes: {100 * n_once / icd_counts.shape[0]:.1f}%")
print(f"share of rows: {100 * n_once / len(model_df):.1f}%")
icd_counts[icd_counts >9]


In [ ]:
# model_df is now fully assembled by cells 68-78 with:
#   RX cost split (RXSF_PATIENT, RXXP_TOTAL), 3-dummy insurance, SEX/RACE/POV/DLAY dummies,
#   raw RXNAME + ICD10CDX (cell 90 will one-hot them at person level).
# Just drop the label column and print a summary.

model_df = model_df.drop(columns=['ICD10CDX_LABEL'], errors='ignore')

print(f"pair-level rows: {len(model_df):,}")
print(f"unique patients: {model_df['DUPERSID'].nunique():,}")
print(f"unique RXNAME: {model_df['RXNAME'].nunique():,}")
print(f"unique ICD10CDX: {model_df['ICD10CDX'].nunique():,}")
print(
    f"insurance rows: private={int(model_df['INSCOV_PRIVATE'].sum()):,} "
    f"public={int(model_df['INSCOV_PUBLIC'].sum()):,} "
    f"uninsured={int(model_df['INSCOV_UNINSURED'].sum()):,}"
)
model_df.head()

In [ ]:
# --- One-hot drugs + conditions (0/1), then 1 row per patient ---
rx_dummies = pd.get_dummies(model_df["RXNAME"], prefix="RX").astype(int)
# Multi-hot ICD dummies from chronic_conditions (comma-joined list of every
# linked chronic ICD, not just the primary one). Falls back to the old
# primary-ICD one-hot when chronic_conditions is missing.
if "chronic_conditions" in model_df.columns:
    icd_dummies = (
        model_df["chronic_conditions"]
        .fillna("")
        .str.get_dummies(sep=",")
        .add_prefix("ICD_")
        .astype(int)
    )
    # drop empty-string column that appears when chronic_conditions is blank
    icd_dummies = icd_dummies.drop(columns=[c for c in ["ICD_"] if c in icd_dummies.columns])
else:
    icd_dummies = pd.get_dummies(model_df["ICD10CDX"], prefix="ICD").astype(int)

pair_df = pd.concat(
    [model_df.drop(columns=["RXNAME", "ICD10CDX"]), rx_dummies, icd_dummies],
    axis=1,
)

rx_cols = [c for c in pair_df.columns if c.startswith("RX_")]
icd_cols = [c for c in pair_df.columns if c.startswith("ICD_")]
COST_COLS = {"RXSF_PATIENT", "RXXP_TOTAL"}
person_cols = [
    c for c in pair_df.columns
    if c not in {"DUPERSID", "meps_adherence_ratio"} | COST_COLS
    and c not in rx_cols and c not in icd_cols
]

agg = {c: "max" for c in person_cols + rx_cols + icd_cols}
agg["meps_adherence_ratio"] = "mean"
agg["RXSF_PATIENT"] = "sum"  # total patient-paid dollars across a patient's fills
agg["RXXP_TOTAL"]   = "sum"  # total drug expenditure across a patient's fills
# Pills/day and strength: average across a patient's drug–condition pairs
for _c in ("medication_freq", "medication_dose"):
    if _c in pair_df.columns:
        agg[_c] = "mean"
for _c in ("medication_dose_unit", "medication_qty_unit"):
    if _c in pair_df.columns:
        agg[_c] = "first"

model_df = pair_df.groupby("DUPERSID", as_index=False).agg(agg)

# fraction of drug bill patient paid out-of-pocket (0..1). NaN-safe.
model_df["PATIENT_COST_SHARE"] = np.where(
    model_df["RXXP_TOTAL"] > 0,
    model_df["RXSF_PATIENT"] / model_df["RXXP_TOTAL"],
    0.0,
)
model_df = model_df.drop(columns=["RXSF_PATIENT", "RXXP_TOTAL"])

model_df["is_adherent"] = np.where(model_df["meps_adherence_ratio"] >= 60, 1, 0)

print(f"patient-level rows: {len(model_df):,} (should equal unique DUPERSID)")
print(f"RX dummy cols: {len(rx_cols)} | ICD dummy cols: {len(icd_cols)}")
print(f"is_adherent value counts:\n{model_df['is_adherent'].value_counts()}")
print("PATIENT_COST_SHARE describe:")
print(model_df['PATIENT_COST_SHARE'].describe().round(3))
model_df.head()

In [ ]:
# Counts + examples: patients with >1 drug and/or >1 condition
rx_cols = [c for c in model_df.columns if c.startswith("RX_")]
icd_cols = [c for c in model_df.columns if c.startswith("ICD_")]

model_df["n_drugs"] = model_df[rx_cols].sum(axis=1)
model_df["n_conditions"] = model_df[icd_cols].sum(axis=1)

n_multi_drug = int((model_df["n_drugs"] > 1).sum())
n_multi_cond = int((model_df["n_conditions"] > 1).sum())
n_both = int(((model_df["n_drugs"] > 1) & (model_df["n_conditions"] > 1)).sum())

print(f"patients total: {len(model_df):,}")
print(f"patients with >1 drug: {n_multi_drug:,}")
print(f"patients with >1 condition: {n_multi_cond:,}")
print(f"patients with >1 drug AND >1 condition: {n_both:,}")

print("\nn_drugs distribution:")
display(model_df["n_drugs"].value_counts().sort_index().head(15))
print("\nn_conditions distribution:")
display(model_df["n_conditions"].value_counts().sort_index().head(15))

# Example patients
example_cols = ["DUPERSID", "n_drugs", "n_conditions", "meps_adherence_ratio", "is_adherent"]

print("\n--- examples: >1 drug ---")
multi_drug_ex = model_df.loc[model_df["n_drugs"] > 1, example_cols].head(5)
display(multi_drug_ex)
for _, row in multi_drug_ex.iterrows():
    pid = row["DUPERSID"]
    drugs = [c.replace("RX_", "") for c in rx_cols if model_df.loc[model_df["DUPERSID"] == pid, c].iloc[0] == 1]
    print(f"DUPERSID {pid}: drugs={drugs}")

print("\n--- examples: >1 condition ---")
multi_cond_ex = model_df.loc[model_df["n_conditions"] > 1, example_cols].head(5)
display(multi_cond_ex)
for _, row in multi_cond_ex.iterrows():
    pid = row["DUPERSID"]
    conds = [c.replace("ICD_", "") for c in icd_cols if model_df.loc[model_df["DUPERSID"] == pid, c].iloc[0] == 1]
    print(f"DUPERSID {pid}: conditions={conds}")


In [ ]:
# ---- Modeling section: consolidated imports ----
# Everything below (RF baseline, RF + n_conditions, XGBoost, XGBoost + TC1S1,
# 3-model comparison, SHAP, plots) uses this same set of imports. Kept in
# one place instead of repeating per cell — Jupyter carries them across cells.

import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import (
    HalvingGridSearchCV, StratifiedKFold, train_test_split,
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, roc_auc_score, RocCurveDisplay,
    precision_score, recall_score,
)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import shap
import time

def plot_cm_with_recall(cm, ax, title, labels=("not adherent", "adherent"), cmap="Blues"):
    """Render a 2x2 confusion matrix with row-normalized recall + accuracy in title.

    Each cell shows ``count\n(row%)`` where row% = 100 * cm[i,j] / cm[i,:].sum()
    (i.e., recall for class i). Title has ``(accuracy: XX.X%)`` appended, where
    accuracy = trace(cm) / sum(cm). Rows sum to 100%.
    """
    cm = np.asarray(cm)
    row_totals = cm.sum(axis=1, keepdims=True)
    pct = np.divide(cm * 100.0, row_totals,
                    out=np.zeros_like(cm, dtype=float),
                    where=row_totals > 0)
    acc = cm.trace() / cm.sum() * 100 if cm.sum() > 0 else 0.0

    im = ax.imshow(cm, cmap=cmap)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{title} (accuracy: {acc:.1f}%)")

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > thresh else "black"
            ax.text(j, i, f"{int(cm[i, j])}\n({pct[i, j]:.1f}%)",
                    ha="center", va="center", color=color, fontsize=11)
    return ax


In [ ]:
# Random Forest: predict is_adherent (hyperparameter tuning)
# Edit PARAM_GRID / CV settings below if you want to change the search.

# Restored full grid (was trimmed for the manual loop). HalvingGridSearchCV
# prunes losers early, so a big search space costs the same as a small one.
# Reference: Koster & Sigrist 2024, "Selecting Hyperparameters for Tree-Boosting"
# (arXiv:2602.05786) — small grids "often yield very inaccurate models".
PARAM_GRID = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
}

CV_FOLDS = 5
SCORING = "f1"
TEST_SIZE = 0.2
RANDOM_STATE = 42

drop_cols = [
    "DUPERSID",
    "is_adherent",
    "meps_adherence_ratio",  # leakage
    "n_drugs",
    "n_conditions",
]

X = model_df.drop(columns=[c for c in drop_cols if c in model_df.columns])
y = model_df["is_adherent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"X shape: {X.shape} | train: {X_train.shape[0]} | test: {X_test.shape[0]}")
print(f"class balance (full):\n{y.value_counts(normalize=True).round(3)}")
print(f"GridSearch combos: {np.prod([len(v) for v in PARAM_GRID.values()])} x {CV_FOLDS} folds")


In [ ]:
# HalvingGridSearchCV — starts with all candidates on a small n_samples subset,
# prunes losers each round, and escalates survivors to more data. Explores the
# full PARAM_GRID at roughly the same wall-clock as a small manual grid loop.

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = HalvingGridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=PARAM_GRID,
    scoring="f1",
    cv=cv,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

t0 = time.perf_counter()
search.fit(X_train, y_train)
print(f"\nDone in {(time.perf_counter() - t0) / 60:.1f} minutes")
print("Best params:", search.best_params_)
print(f"Best CV f1: {search.best_score_:.4f}")
print(f"Iterations run: {search.n_iterations_} | "
      f"n_candidates per iter: {search.n_candidates_} | "
      f"n_resources per iter: {search.n_resources_}")

best_rf = search.best_estimator_
y_pred = best_rf.predict(X_test)
print("\nClassification report (test set):")
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix (test set):")
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
plot_cm_with_recall(cm, ax, "Random Forest (F1-tuned) — is_adherent")
plt.tight_layout()
plt.show()

# Legacy aliases so downstream cells that grew around the old manual-loop
# variable names keep working without an edit.
best_params = _tuned_local = {k: best_rf.get_params()[k] for k in PARAM_GRID if k in best_rf.get_params()}
best_score = float(search.best_score_)


In [ ]:
# HalvingGridSearchCV — starts with all candidates on a small n_samples subset,
# prunes losers each round, and escalates survivors to more data. Explores the
# full PARAM_GRID at roughly the same wall-clock as a small manual grid loop.

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = HalvingGridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=PARAM_GRID,
    scoring="roc_auc",
    cv=cv,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

t0 = time.perf_counter()
search.fit(X_train, y_train)
print(f"\nDone in {(time.perf_counter() - t0) / 60:.1f} minutes")
print("Best params:", search.best_params_)
print(f"Best CV roc_auc: {search.best_score_:.4f}")
print(f"Iterations run: {search.n_iterations_} | "
      f"n_candidates per iter: {search.n_candidates_} | "
      f"n_resources per iter: {search.n_resources_}")

best_rf_auc = search.best_estimator_
y_pred = best_rf_auc.predict(X_test)
y_proba = best_rf_auc.predict_proba(X_test)[:, 1]

print(f"\nTest ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification report (test set):")
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix (test set):")
print(cm)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_cm_with_recall(cm, axes[0], "Random Forest (AUC-tuned) — confusion matrix")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title("Random Forest (AUC-tuned) — ROC curve")
plt.tight_layout()
plt.show()

best_params_auc = {k: best_rf_auc.get_params()[k] for k in PARAM_GRID if k in best_rf_auc.get_params()}
best_score_auc = float(search.best_score_)


In [ ]:
# Feature importance: Gini (RF impurity decrease) + SHAP
# Uses AUC-tuned model if available, else F1-tuned model.

model_for_imp = best_rf_auc if "best_rf_auc" in dir() else best_rf
model_label = "AUC-tuned" if "best_rf_auc" in dir() and model_for_imp is best_rf_auc else "F1-tuned"

TOP_N = 20  # edit: how many features to show

gini_imp = (
    pd.Series(model_for_imp.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

print(f"Gini importance — top {TOP_N} ({model_label} RF)")
display(gini_imp.head(TOP_N).to_frame("gini_importance"))

fig, ax = plt.subplots(figsize=(8, 6))
gini_imp.head(TOP_N).iloc[::-1].plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Gini importance (mean decrease in impurity)")
ax.set_title(f"Top {TOP_N} features — Gini ({model_label})")
plt.tight_layout()
plt.show()


In [ ]:
# SHAP values (TreeExplainer) — mean |SHAP| ranking + beeswarm
# Uses a sample of the training set for speed; raise SHAP_SAMPLE if you want more precision.

SHAP_SAMPLE = 500  # edit: number of rows for SHAP

X_shap = X_train.sample(n=min(SHAP_SAMPLE, len(X_train)), random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(model_for_imp)
shap_values = explainer.shap_values(X_shap)

# Binary RF: shap_values is often [class0, class1] or a 3D array depending on shap version
if isinstance(shap_values, list):
    shap_pos = shap_values[1]  # class 1 = adherent
elif getattr(shap_values, "ndim", 0) == 3:
    shap_pos = shap_values[:, :, 1]
else:
    shap_pos = shap_values

mean_abs_shap = (
    pd.Series(np.abs(shap_pos).mean(axis=0), index=X_shap.columns)
    .sort_values(ascending=False)
)

print(f"SHAP mean |value| — top {TOP_N} ({model_label} RF, n={len(X_shap)})")
display(mean_abs_shap.head(TOP_N).to_frame("mean_abs_shap"))

fig, ax = plt.subplots(figsize=(8, 6))
mean_abs_shap.head(TOP_N).iloc[::-1].plot(kind="barh", ax=ax, color="darkorange")
ax.set_xlabel("mean |SHAP value|")
ax.set_title(f"Top {TOP_N} features — SHAP ({model_label})")
plt.tight_layout()
plt.show()

print("SHAP beeswarm (class = adherent):")
shap.summary_plot(shap_pos, X_shap, max_display=TOP_N, show=False)
plt.title(f"SHAP beeswarm — {model_label}")
plt.tight_layout()
plt.show()


In [ ]:
# F1-tuned RF — Gini feature importance
TOP_N_F1 = 20  # edit

gini_imp_f1 = (
    pd.Series(best_rf.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

# Read best params off the fitted estimator (rather than relying on a
# `best_params` variable from a manual grid loop — HalvingGridSearchCV
# fits the winner directly to best_rf).
_p = best_rf.get_params()
_tuned = {k: _p[k] for k in PARAM_GRID if k in _p}

print(f"Gini importance — top {TOP_N_F1} (F1-tuned RF)")
print("Best F1 params:", _tuned)
display(gini_imp_f1.head(TOP_N_F1).to_frame("gini_importance"))

fig, ax = plt.subplots(figsize=(8, 6))
gini_imp_f1.head(TOP_N_F1).iloc[::-1].plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Gini importance (mean decrease in impurity)")
ax.set_title(f"Top {TOP_N_F1} features — Gini (F1-tuned)")
plt.tight_layout()
plt.show()


In [ ]:
# F1-tuned RF — SHAP feature importance

SHAP_SAMPLE_F1 = 500  # edit

X_shap_f1 = X_train.sample(n=min(SHAP_SAMPLE_F1, len(X_train)), random_state=RANDOM_STATE)

explainer_f1 = shap.TreeExplainer(best_rf)
shap_values_f1 = explainer_f1.shap_values(X_shap_f1)

if isinstance(shap_values_f1, list):
    shap_pos_f1 = shap_values_f1[1]
elif getattr(shap_values_f1, "ndim", 0) == 3:
    shap_pos_f1 = shap_values_f1[:, :, 1]
else:
    shap_pos_f1 = shap_values_f1

mean_abs_shap_f1 = (
    pd.Series(np.abs(shap_pos_f1).mean(axis=0), index=X_shap_f1.columns)
    .sort_values(ascending=False)
)

print(f"SHAP mean |value| — top {TOP_N_F1} (F1-tuned RF, n={len(X_shap_f1)})")
display(mean_abs_shap_f1.head(TOP_N_F1).to_frame("mean_abs_shap"))

fig, ax = plt.subplots(figsize=(8, 6))
mean_abs_shap_f1.head(TOP_N_F1).iloc[::-1].plot(kind="barh", ax=ax, color="darkorange")
ax.set_xlabel("mean |SHAP value|")
ax.set_title(f"Top {TOP_N_F1} features — SHAP (F1-tuned)")
plt.tight_layout()
plt.show()

print("SHAP beeswarm (class = adherent, F1-tuned):")
shap.summary_plot(shap_pos_f1, X_shap_f1, max_display=TOP_N_F1, show=False)
plt.title("SHAP beeswarm — F1-tuned")
plt.tight_layout()
plt.show()


In [ ]:
# RF with n_drugs + n_conditions included as features
# (still drops meps_adherence_ratio to avoid leakage)

# Ensure count columns exist on patient-level model_df
rx_cols = [c for c in model_df.columns if c.startswith("RX_")]
icd_cols = [c for c in model_df.columns if c.startswith("ICD_")]
model_df["n_drugs"] = model_df[rx_cols].sum(axis=1)
model_df["n_conditions"] = model_df[icd_cols].sum(axis=1)

# Restored full grid (halving handles it — see PARAM_GRID above).
PARAM_GRID_NC = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
}

CV_FOLDS = 5
TEST_SIZE = 0.2
RANDOM_STATE = 42

drop_cols_nc = [
    "DUPERSID",
    "is_adherent",
    "meps_adherence_ratio",  # leakage — still dropped
    # n_drugs / n_conditions KEPT as features
]

X_nc = model_df.drop(columns=[c for c in drop_cols_nc if c in model_df.columns])
y_nc = model_df["is_adherent"]

X_train_nc, X_test_nc, y_train_nc, y_test_nc = train_test_split(
    X_nc, y_nc, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_nc
)

print("Features include n_drugs / n_conditions:", 
      "n_drugs" in X_nc.columns and "n_conditions" in X_nc.columns)
print(f"X_nc shape: {X_nc.shape} | train: {X_train_nc.shape[0]} | test: {X_test_nc.shape[0]}")
print(f"GridSearch combos: {np.prod([len(v) for v in PARAM_GRID_NC.values()])} x {CV_FOLDS} folds")


In [ ]:
# HalvingGridSearchCV — starts with all candidates on a small n_samples subset,
# prunes losers each round, and escalates survivors to more data. Explores the
# full PARAM_GRID_NC at roughly the same wall-clock as a small manual grid loop.

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = HalvingGridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=PARAM_GRID_NC,
    scoring="f1",
    cv=cv,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

t0 = time.perf_counter()
search.fit(X_train_nc, y_train_nc)
print(f"\nDone in {(time.perf_counter() - t0) / 60:.1f} minutes")
print("Best params:", search.best_params_)
print(f"Best CV f1: {search.best_score_:.4f}")
print(f"Iterations run: {search.n_iterations_} | "
      f"n_candidates per iter: {search.n_candidates_} | "
      f"n_resources per iter: {search.n_resources_}")

best_rf_nc = search.best_estimator_
y_pred = best_rf_nc.predict(X_test_nc)
print("\nClassification report (test set):")
print(classification_report(y_test_nc, y_pred, digits=3))

cm = confusion_matrix(y_test_nc, y_pred)
print("Confusion matrix (test set):")
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
plot_cm_with_recall(cm, ax, "RF (F1) — with n_drugs & n_conditions")
plt.tight_layout()
plt.show()

best_params_nc = {k: best_rf_nc.get_params()[k] for k in PARAM_GRID_NC if k in best_rf_nc.get_params()}
best_score_nc = float(search.best_score_)


In [ ]:
# HalvingGridSearchCV — starts with all candidates on a small n_samples subset,
# prunes losers each round, and escalates survivors to more data. Explores the
# full PARAM_GRID_NC at roughly the same wall-clock as a small manual grid loop.

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = HalvingGridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=PARAM_GRID_NC,
    scoring="roc_auc",
    cv=cv,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

t0 = time.perf_counter()
search.fit(X_train_nc, y_train_nc)
print(f"\nDone in {(time.perf_counter() - t0) / 60:.1f} minutes")
print("Best params:", search.best_params_)
print(f"Best CV roc_auc: {search.best_score_:.4f}")
print(f"Iterations run: {search.n_iterations_} | "
      f"n_candidates per iter: {search.n_candidates_} | "
      f"n_resources per iter: {search.n_resources_}")

best_rf_nc_auc = search.best_estimator_
y_pred = best_rf_nc_auc.predict(X_test_nc)
y_proba = best_rf_nc_auc.predict_proba(X_test_nc)[:, 1]

print(f"\nTest ROC-AUC: {roc_auc_score(y_test_nc, y_proba):.4f}")
print("\nClassification report (test set):")
print(classification_report(y_test_nc, y_pred, digits=3))

cm = confusion_matrix(y_test_nc, y_pred)
print("Confusion matrix (test set):")
print(cm)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_cm_with_recall(cm, axes[0], "RF (AUC) — with n_drugs & n_conditions — confusion matrix")
RocCurveDisplay.from_predictions(y_test_nc, y_proba, ax=axes[1])
axes[1].set_title("RF (AUC) — with n_drugs & n_conditions — ROC curve")
plt.tight_layout()
plt.show()

best_params_nc_auc = {k: best_rf_nc_auc.get_params()[k] for k in PARAM_GRID_NC if k in best_rf_nc_auc.get_params()}
best_score_nc_auc = float(search.best_score_)


In [ ]:
# New modeling frame WITHOUT RXNAME / RX_ one-hot columns
# Keeps demographics, ICD dummies, n_drugs/n_conditions (if present), adherence target, etc.

rx_cols = [c for c in model_df.columns if c.startswith("RX_")]
#icd_cols = [c for c in model_df.columns if c.startswith("ICD_")]

# Ensure count columns exist before dropping drug dummies
if "n_drugs" not in model_df.columns and rx_cols:
    model_df["n_drugs"] = model_df[rx_cols].sum(axis=1)
#if "n_conditions" not in model_df.columns and icd_cols:
#    model_df["n_conditions"] = model_df[icd_cols].sum(axis=1)

drop_rx = rx_cols + [c for c in ["RXNAME"] if c in model_df.columns]
model_df_no_rx = model_df.drop(columns=drop_rx).copy()

print(f"original model_df shape: {model_df.shape}")
print(f"model_df_no_rx shape:    {model_df_no_rx.shape}")
print(f"dropped {len(drop_rx)} RXNAME/RX_ columns")
print(f"remaining columns ({len(model_df_no_rx.columns)}):")
print(model_df_no_rx.columns.tolist())
model_df_no_rx.head()


In [ ]:
# XGBoost: predict is_adherent using model_df_no_rx (no RXNAME / RX_ dummies)
# Edit XGB_PARAM_GRID below to change the search.
# If import fails on macOS with libomp.dylib missing, run: brew install libomp

# Restored full grid. HalvingGridSearchCV prunes losers early so a wide
# search costs about the same as a small one. Kept all axes because
# Koster & Sigrist (2024) show every XGBoost hyperparameter can affect
# accuracy on tabular data.
XGB_PARAM_GRID = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "reg_lambda": [1.0, 5.0],
}

CV_FOLDS = 5
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Ensure model_df_no_rx exists
if "model_df_no_rx" not in dir():
    rx_cols = [c for c in model_df.columns if c.startswith("RX_")]
    icd_cols = [c for c in model_df.columns if c.startswith("ICD_")]
    if "n_drugs" not in model_df.columns and rx_cols:
        model_df["n_drugs"] = model_df[rx_cols].sum(axis=1)
    if "n_conditions" not in model_df.columns and icd_cols:
        model_df["n_conditions"] = model_df[icd_cols].sum(axis=1)
    drop_rx = rx_cols + [c for c in ["RXNAME"] if c in model_df.columns]
    model_df_no_rx = model_df.drop(columns=drop_rx).copy()

drop_cols_xgb = [
    "DUPERSID",
    "is_adherent",
    "meps_adherence_ratio",  # leakage
]

X_xgb = model_df_no_rx.drop(columns=[c for c in drop_cols_xgb if c in model_df_no_rx.columns])
y_xgb = model_df_no_rx["is_adherent"]

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgb, y_xgb, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_xgb
)

# class imbalance helper for XGBoost
neg, pos = np.bincount(y_train_xgb.astype(int))
scale_pos_weight = neg / pos if pos > 0 else 1.0

print(f"X_xgb shape: {X_xgb.shape} | train: {X_train_xgb.shape[0]} | test: {X_test_xgb.shape[0]}")
print(f"class balance (train):\n{y_train_xgb.value_counts(normalize=True).round(3)}")
print(f"scale_pos_weight: {scale_pos_weight:.3f}")
print(f"Grid combos: {np.prod([len(v) for v in XGB_PARAM_GRID.values()])} x {CV_FOLDS} folds")
print(f"features ({X_xgb.shape[1]}): {X_xgb.columns.tolist()[:20]}...")


In [ ]:
# HalvingGridSearchCV — starts with all candidates on a small n_samples subset,
# prunes losers each round, and escalates survivors to more data. Explores the
# full XGB_PARAM_GRID at roughly the same wall-clock as a small manual grid loop.

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = HalvingGridSearchCV(
    estimator=XGBClassifier(objective="binary:logistic", eval_metric="auc" if "roc" in "f1" else "logloss", random_state=RANDOM_STATE, n_jobs=1, tree_method="hist", scale_pos_weight=scale_pos_weight, verbosity=0),
    param_grid=XGB_PARAM_GRID,
    scoring="f1",
    cv=cv,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

t0 = time.perf_counter()
search.fit(X_train_xgb, y_train_xgb)
print(f"\nDone in {(time.perf_counter() - t0) / 60:.1f} minutes")
print("Best params:", search.best_params_)
print(f"Best CV f1: {search.best_score_:.4f}")
print(f"Iterations run: {search.n_iterations_} | "
      f"n_candidates per iter: {search.n_candidates_} | "
      f"n_resources per iter: {search.n_resources_}")

best_xgb = search.best_estimator_
y_pred = best_xgb.predict(X_test_xgb)
print("\nClassification report (test set):")
print(classification_report(y_test_xgb, y_pred, digits=3))

cm = confusion_matrix(y_test_xgb, y_pred)
print("Confusion matrix (test set):")
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
plot_cm_with_recall(cm, ax, "XGBoost (F1) — no RX dummies")
plt.tight_layout()
plt.show()

best_params_xgb = {k: best_xgb.get_params()[k] for k in XGB_PARAM_GRID if k in best_xgb.get_params()}
best_score_xgb = float(search.best_score_)


In [ ]:
# HalvingGridSearchCV — starts with all candidates on a small n_samples subset,
# prunes losers each round, and escalates survivors to more data. Explores the
# full XGB_PARAM_GRID at roughly the same wall-clock as a small manual grid loop.

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = HalvingGridSearchCV(
    estimator=XGBClassifier(objective="binary:logistic", eval_metric="auc" if "roc" in "roc_auc" else "logloss", random_state=RANDOM_STATE, n_jobs=1, tree_method="hist", scale_pos_weight=scale_pos_weight, verbosity=0),
    param_grid=XGB_PARAM_GRID,
    scoring="roc_auc",
    cv=cv,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

t0 = time.perf_counter()
search.fit(X_train_xgb, y_train_xgb)
print(f"\nDone in {(time.perf_counter() - t0) / 60:.1f} minutes")
print("Best params:", search.best_params_)
print(f"Best CV roc_auc: {search.best_score_:.4f}")
print(f"Iterations run: {search.n_iterations_} | "
      f"n_candidates per iter: {search.n_candidates_} | "
      f"n_resources per iter: {search.n_resources_}")

best_xgb_auc = search.best_estimator_
y_pred = best_xgb_auc.predict(X_test_xgb)
y_proba = best_xgb_auc.predict_proba(X_test_xgb)[:, 1]

print(f"\nTest ROC-AUC: {roc_auc_score(y_test_xgb, y_proba):.4f}")
print("\nClassification report (test set):")
print(classification_report(y_test_xgb, y_pred, digits=3))

cm = confusion_matrix(y_test_xgb, y_pred)
print("Confusion matrix (test set):")
print(cm)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_cm_with_recall(cm, axes[0], "XGBoost (AUC) — no RX dummies — confusion matrix")
RocCurveDisplay.from_predictions(y_test_xgb, y_proba, ax=axes[1])
axes[1].set_title("XGBoost (AUC) — no RX dummies — ROC curve")
plt.tight_layout()
plt.show()

best_params_xgb_auc = {k: best_xgb_auc.get_params()[k] for k in XGB_PARAM_GRID if k in best_xgb_auc.get_params()}
best_score_xgb_auc = float(search.best_score_)


In [ ]:
# XGBoost feature importance — Gain (Gini-style) + SHAP
# XGBoost — Gain feature importance (Gini-style impurity decrease analogue)
# Requires best_xgb / best_xgb_auc and X_train_xgb from the XGBoost cells above.

model_xgb_imp = best_xgb_auc if "best_xgb_auc" in dir() else best_xgb
model_xgb_label = (
    "AUC-tuned"
    if "best_xgb_auc" in dir() and model_xgb_imp is best_xgb_auc
    else "F1-tuned"
)

TOP_N_XGB = 20  # edit: how many features to show

# sklearn XGBClassifier.feature_importances_ == gain by default
gain_imp_xgb = (
    pd.Series(model_xgb_imp.feature_importances_, index=X_train_xgb.columns)
    .sort_values(ascending=False)
)

# Best params come off the fitted estimator (HalvingGridSearchCV refits the
# winner directly, so best_xgb / best_xgb_auc carry the tuned values).
_p_xgb = model_xgb_imp.get_params()
_tuned_xgb = {k: _p_xgb[k] for k in XGB_PARAM_GRID if k in _p_xgb}

print(f"Gain importance — top {TOP_N_XGB} ({model_xgb_label} XGBoost)")
print("Best params:", _tuned_xgb)
display(gain_imp_xgb.head(TOP_N_XGB).to_frame("gain_importance"))

fig, ax = plt.subplots(figsize=(8, 6))
gain_imp_xgb.head(TOP_N_XGB).iloc[::-1].plot(kind="barh", ax=ax, color="seagreen")
ax.set_xlabel("Gain importance (mean loss reduction from splits)")
ax.set_title(f"Top {TOP_N_XGB} features — Gain / Gini-style ({model_xgb_label} XGBoost)")
plt.tight_layout()
plt.show()


In [ ]:
# XGBoost — SHAP (TreeExplainer): mean |SHAP| ranking + beeswarm
# Uses a sample of the training set for speed; raise SHAP_SAMPLE_XGB for more precision.

SHAP_SAMPLE_XGB = 500  # edit

X_shap_xgb = X_train_xgb.sample(
    n=min(SHAP_SAMPLE_XGB, len(X_train_xgb)), random_state=RANDOM_STATE
)

explainer_xgb = shap.TreeExplainer(model_xgb_imp)
shap_values_xgb = explainer_xgb.shap_values(X_shap_xgb)

# Binary XGB: usually a 2D array for the positive class (adherent)
if isinstance(shap_values_xgb, list):
    shap_pos_xgb = shap_values_xgb[1]
elif getattr(shap_values_xgb, "ndim", 0) == 3:
    shap_pos_xgb = shap_values_xgb[:, :, 1]
else:
    shap_pos_xgb = shap_values_xgb

mean_abs_shap_xgb = (
    pd.Series(np.abs(shap_pos_xgb).mean(axis=0), index=X_shap_xgb.columns)
    .sort_values(ascending=False)
)

print(
    f"SHAP mean |value| — top {TOP_N_XGB} "
    f"({model_xgb_label} XGBoost, n={len(X_shap_xgb)})"
)
display(mean_abs_shap_xgb.head(TOP_N_XGB).to_frame("mean_abs_shap"))

fig, ax = plt.subplots(figsize=(8, 6))
mean_abs_shap_xgb.head(TOP_N_XGB).iloc[::-1].plot(kind="barh", ax=ax, color="darkorange")
ax.set_xlabel("mean |SHAP value|")
ax.set_title(f"Top {TOP_N_XGB} features — SHAP ({model_xgb_label} XGBoost)")
plt.tight_layout()
plt.show()

print("SHAP beeswarm (class = adherent):")
shap.summary_plot(shap_pos_xgb, X_shap_xgb, max_display=TOP_N_XGB, show=False)
plt.title(f"SHAP beeswarm — {model_xgb_label} XGBoost")
plt.tight_layout()
plt.show()


In [ ]:
# XGBoost with TC1S1 one-hots (`model_df_no_rx_tc`)
# Build model_df_no_rx_tc: model_df_no_rx + one-hot TC1S1 (patient level)
# TC1S1 comes from pair-level new_grouped_merge_df; a patient gets 1 if any of their drugs has that subclass.

if "model_df_no_rx" not in dir():
    raise NameError("Run the model_df_no_rx cell first.")
if "new_grouped_merge_df" not in dir() or "TC1S1" not in new_grouped_merge_df.columns:
    raise NameError("Need new_grouped_merge_df with TC1S1 (from earlier merge cells).")

tc = new_grouped_merge_df[["DUPERSID", "TC1S1"]].copy()
tc["TC1S1"] = pd.to_numeric(tc["TC1S1"], errors="coerce")
# Drop MEPS missing/inapplicable sentinels (negatives) and NaNs
tc = tc[tc["TC1S1"].notna() & (tc["TC1S1"] > 0)].copy()
tc["TC1S1"] = tc["TC1S1"].astype(int)

tc_dummies = pd.get_dummies(tc["TC1S1"].astype(str), prefix="TC1S1").astype(int)
tc_patient = (
    pd.concat([tc[["DUPERSID"]], tc_dummies], axis=1)
    .groupby("DUPERSID", as_index=False)
    .max()
)

model_df_no_rx_tc = model_df_no_rx.merge(tc_patient, on="DUPERSID", how="left")
tc_cols = [c for c in model_df_no_rx_tc.columns if c.startswith("TC1S1_")]
model_df_no_rx_tc[tc_cols] = model_df_no_rx_tc[tc_cols].fillna(0).astype(int)

print(f"model_df_no_rx shape:    {model_df_no_rx.shape}")
print(f"model_df_no_rx_tc shape: {model_df_no_rx_tc.shape}")
print(f"TC1S1 one-hot columns:   {len(tc_cols)}")
print(f"patients with ≥1 TC1S1:  {(model_df_no_rx_tc[tc_cols].sum(axis=1) > 0).sum():,}")
print("TC1S1_* prevalence (top 15):")
display(
    model_df_no_rx_tc[tc_cols]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("n_patients")
)
model_df_no_rx_tc.head()


In [ ]:
# XGBoost setup on model_df_no_rx_tc (no RX dummies + TC1S1 one-hots)

# Reuse grid from earlier XGB cells if present; otherwise define here
if "XGB_PARAM_GRID" not in dir():
# Restored full grid (halving handles it).
    XGB_PARAM_GRID = {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7, 10],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "min_child_weight": [1, 5],
        "reg_lambda": [1.0, 5.0],
    }
if "CV_FOLDS" not in dir():
    CV_FOLDS = 5
if "TEST_SIZE" not in dir():
    TEST_SIZE = 0.2
if "RANDOM_STATE" not in dir():
    RANDOM_STATE = 42

drop_cols_xgb_tc = [
    "DUPERSID",
    "is_adherent",
    "meps_adherence_ratio",  # leakage
]

X_xgb_tc = model_df_no_rx_tc.drop(
    columns=[c for c in drop_cols_xgb_tc if c in model_df_no_rx_tc.columns]
)
y_xgb_tc = model_df_no_rx_tc["is_adherent"]

X_train_xgb_tc, X_test_xgb_tc, y_train_xgb_tc, y_test_xgb_tc = train_test_split(
    X_xgb_tc, y_xgb_tc, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_xgb_tc
)

neg_tc, pos_tc = np.bincount(y_train_xgb_tc.astype(int))
scale_pos_weight_tc = neg_tc / pos_tc if pos_tc > 0 else 1.0

print(
    f"X_xgb_tc shape: {X_xgb_tc.shape} | "
    f"train: {X_train_xgb_tc.shape[0]} | test: {X_test_xgb_tc.shape[0]}"
)
print(f"class balance (train):\n{y_train_xgb_tc.value_counts(normalize=True).round(3)}")
print(f"scale_pos_weight: {scale_pos_weight_tc:.3f}")
print(
    f"Grid combos: {np.prod([len(v) for v in XGB_PARAM_GRID.values()])} x {CV_FOLDS} folds"
)
tc_feat = [c for c in X_xgb_tc.columns if c.startswith("TC1S1_")]
print(f"features: {X_xgb_tc.shape[1]} total | {len(tc_feat)} TC1S1_*")
print(f"feature sample: {X_xgb_tc.columns.tolist()[:15]}...")


In [ ]:
# HalvingGridSearchCV — starts with all candidates on a small n_samples subset,
# prunes losers each round, and escalates survivors to more data. Explores the
# full XGB_PARAM_GRID at roughly the same wall-clock as a small manual grid loop.

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = HalvingGridSearchCV(
    estimator=XGBClassifier(objective="binary:logistic", eval_metric="auc" if "roc" in "f1" else "logloss", random_state=RANDOM_STATE, n_jobs=1, tree_method="hist", scale_pos_weight=scale_pos_weight_tc, verbosity=0),
    param_grid=XGB_PARAM_GRID,
    scoring="f1",
    cv=cv,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

t0 = time.perf_counter()
search.fit(X_train_xgb_tc, y_train_xgb_tc)
print(f"\nDone in {(time.perf_counter() - t0) / 60:.1f} minutes")
print("Best params:", search.best_params_)
print(f"Best CV f1: {search.best_score_:.4f}")
print(f"Iterations run: {search.n_iterations_} | "
      f"n_candidates per iter: {search.n_candidates_} | "
      f"n_resources per iter: {search.n_resources_}")

best_xgb_tc = search.best_estimator_
y_pred = best_xgb_tc.predict(X_test_xgb_tc)
print("\nClassification report (test set):")
print(classification_report(y_test_xgb_tc, y_pred, digits=3))

cm = confusion_matrix(y_test_xgb_tc, y_pred)
print("Confusion matrix (test set):")
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
plot_cm_with_recall(cm, ax, "XGBoost + TC1S1 (F1)")
plt.tight_layout()
plt.show()

best_params_xgb_tc = {k: best_xgb_tc.get_params()[k] for k in XGB_PARAM_GRID if k in best_xgb_tc.get_params()}
best_score_xgb_tc = float(search.best_score_)


In [ ]:
# MARRYXX / REGIONXX (year-end + backfill) + EDUCYR → merge onto model_df
# MARRY: 1 married … 10 separated-in-round; negatives -1/-7/-8
# REGION: 1 NE, 2 MW, 3 S, 4 W; -1 inapplicable
# EDUCYR: 0–17 years; negatives -1/-7/-8

def _backfill_newest_to_oldest(df, cols_newest_to_oldest, out_name):
    """First non-negative value walking newest → oldest rounds."""
    series = [pd.to_numeric(df[c], errors="coerce") for c in cols_newest_to_oldest]
    out = series[0].copy()
    for s in series[1:]:
        need = out.isna() | (out < 0)
        out = out.where(~need, s)
    df[out_name] = out
    return df


demo_extra = demo_df[
    [
        "DUPERSID",
        "EDUCYR",
        "MARRY23X",
        "MARRY53X",
        "MARRY42X",
        "MARRY31X",
        "REGION23",
        "REGION53",
        "REGION42",
        "REGION31",
    ]
].drop_duplicates("DUPERSID").copy()

demo_extra = _backfill_newest_to_oldest(
    demo_extra,
    ["MARRY23X", "MARRY53X", "MARRY42X", "MARRY31X"],
    "MARRYXX",
)
demo_extra = _backfill_newest_to_oldest(
    demo_extra,
    ["REGION23", "REGION53", "REGION42", "REGION31"],
    "REGIONXX",
)
demo_extra["EDUCYR"] = pd.to_numeric(demo_extra["EDUCYR"], errors="coerce")

keep = demo_extra[["DUPERSID", "MARRYXX", "REGIONXX", "EDUCYR"]].copy()

# Attach to model_df (patient-level)
model_df = model_df.drop(columns=["MARRYXX", "REGIONXX", "EDUCYR"], errors="ignore").merge(
    keep, on="DUPERSID", how="left"
)

# Keep downstream frames in sync if they already exist
for _name in ("model_df_no_rx", "model_df_no_rx_tc"):
    if _name in dir():
        globals()[_name] = (
            globals()[_name]
            .drop(columns=["MARRYXX", "REGIONXX", "EDUCYR"], errors="ignore")
            .merge(keep, on="DUPERSID", how="left")
        )

print(f"model_df rows: {len(model_df):,} | patients: {model_df['DUPERSID'].nunique():,}")
for col in ("MARRYXX", "REGIONXX", "EDUCYR"):
    s = pd.to_numeric(model_df[col], errors="coerce")
    n_neg = int((s < 0).sum())
    n_na = int(s.isna().sum())
    print(f"{col}: negatives={n_neg:,} | NaN={n_na:,}")
    if n_neg:
        print(s[s < 0].value_counts().sort_index().to_string())

print("\nValue counts (non-negative):")
display(model_df["MARRYXX"].value_counts(dropna=False).sort_index().to_frame("n"))
display(model_df["REGIONXX"].value_counts(dropna=False).sort_index().to_frame("n"))
print("EDUCYR describe (non-negative only):")
print(model_df.loc[model_df["EDUCYR"] >= 0, "EDUCYR"].describe().round(2))
model_df[["DUPERSID", "MARRYXX", "REGIONXX", "EDUCYR"]].head()


In [ ]:
# medication_freq = pills/day ≈ RXQUANTY / RXDAYSUP
# medication_dose = strength per unit ≈ RXSTRENG (+ medication_dose_unit from RXSTRUNT)
# medication_qty_unit = RXFORM (TABS/CAPS/…) — unit that RXQUANTY is counted in
# Example: 90 TABS / 45 days = 2 per day; dose 10 MG

MAX_PILLS_PER_DAY = 10  # edit: freqs above this → missing (not clipped)

_rx_cols = [
    "DUPERSID", "DRUGIDX", "RXNAME",
    "RXSTRENG", "RXSTRUNT", "RXQUANTY", "RXFORM", "RXDAYSUP",
]
if "df_248" in dir() and set(_rx_cols).issubset(df_248.columns):
    _rx = df_248[_rx_cols].copy()
else:
    _rx = pd.read_excel(MEPS_DIR / "h248a.xlsx", usecols=_rx_cols, engine="calamine")

_dose = pd.to_numeric(_rx["RXSTRENG"], errors="coerce")
_qty = pd.to_numeric(_rx["RXQUANTY"], errors="coerce")
_days = pd.to_numeric(_rx["RXDAYSUP"], errors="coerce")

_dose = _dose.where(_dose >= 0)
_qty_ok = _qty.where(_qty > 0)
_days_ok = _days.where((_days > 0) & (_days < 990))

_freq = _qty_ok / _days_ok
n_before = int(_freq.notna().sum())
n_implausible = int((_freq > MAX_PILLS_PER_DAY).sum())
_freq = _freq.where((_freq > 0) & (_freq <= MAX_PILLS_PER_DAY))

# Units: treat MEPS -15 / blanks as missing
def _clean_unit(s):
    out = s.astype(str).str.strip()
    bad = out.str.lower().isin({"-15", "nan", "none", "", "<na>"})
    return out.where(~bad)

_rx["medication_dose"] = _dose
_rx["medication_dose_unit"] = _clean_unit(_rx["RXSTRUNT"])
_rx["medication_freq"] = _freq
_rx["medication_qty_unit"] = _clean_unit(_rx["RXFORM"])

print(
    f"Fill-level freq: {n_before:,} calculable → "
    f"{n_implausible:,} dropped as >{MAX_PILLS_PER_DAY}/day → "
    f"{int(_freq.notna().sum()):,} kept"
)
print("Top dose units:", _rx["medication_dose_unit"].value_counts().head(8).to_dict())
print("Top qty units:", _rx["medication_qty_unit"].value_counts().head(8).to_dict())


def _first_non_null(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else pd.NA


med_by_drug = (
    _rx.groupby(["DUPERSID", "RXNAME"], as_index=False)
    .agg(
        medication_freq=("medication_freq", "mean"),
        medication_dose=("medication_dose", "mean"),
        medication_dose_unit=("medication_dose_unit", _first_non_null),
        medication_qty_unit=("medication_qty_unit", _first_non_null),
    )
)

med_by_person = (
    med_by_drug.groupby("DUPERSID", as_index=False)
    .agg(
        medication_freq=("medication_freq", "mean"),
        medication_dose=("medication_dose", "mean"),
        medication_dose_unit=("medication_dose_unit", _first_non_null),
        medication_qty_unit=("medication_qty_unit", _first_non_null),
    )
)

_med_cols = [
    "medication_freq",
    "medication_dose",
    "medication_dose_unit",
    "medication_qty_unit",
]
model_df = model_df.drop(columns=_med_cols, errors="ignore")

if "RXNAME" in model_df.columns:
    model_df = model_df.merge(med_by_drug, on=["DUPERSID", "RXNAME"], how="left")
    level = "pair (DUPERSID × RXNAME)"
else:
    model_df = model_df.merge(med_by_person, on="DUPERSID", how="left")
    level = "patient (mean across drugs)"

if "new_grouped_merge_df" in dir() and "RXNAME" in new_grouped_merge_df.columns:
    new_grouped_merge_df = new_grouped_merge_df.drop(columns=_med_cols, errors="ignore").merge(
        med_by_drug, on=["DUPERSID", "RXNAME"], how="left"
    )

for _name in ("model_df_no_rx", "model_df_no_rx_tc"):
    if _name in dir():
        globals()[_name] = (
            globals()[_name]
            .drop(columns=_med_cols, errors="ignore")
            .merge(med_by_person, on="DUPERSID", how="left")
        )

print(f"Merged at {level} → model_df shape {model_df.shape}")
for c in _med_cols:
    print(f"  {c}: non-null {model_df[c].notna().sum():,} / {len(model_df):,}")

print("\nmedication_freq describe:")
print(model_df["medication_freq"].describe(percentiles=[0.5, 0.9, 0.99]).round(3))
display(
    model_df[
        [
            c
            for c in [
                "DUPERSID",
                "RXNAME",
                "medication_freq",
                "medication_qty_unit",
                "medication_dose",
                "medication_dose_unit",
            ]
            if c in model_df.columns
        ]
    ].head(10)
)


In [ ]:
# --- Paper-driven enrichments: ethnicity + dosing-frequency bins ---
# Adds two columns the scoping review + paper 3 flagged as strong predictors
# but that the notebook was not yet using:
#   RACETHX             : MEPS combined race/ethnicity code (1=Hispanic,
#                         2=NH White, 3=NH Black, 4=NH Asian, 5=NH other).
#   medication_freq_bin : bucketed pills-per-day. Paper 3 (Toy et al., COPD)
#                         found PDC drops 43% -> 23% as dosing frequency rises.
# These join model_df, model_df_no_rx, and model_df_no_rx_tc so all three
# downstream model frames pick them up.

# --- 1. RACETHX ---
if "RACETHX" in demo_df.columns:
    _eth = demo_df[["DUPERSID", "RACETHX"]].drop_duplicates("DUPERSID")
    model_df = model_df.drop(columns=["RACETHX"], errors="ignore").merge(
        _eth, on="DUPERSID", how="left"
    )
    for _name in ("model_df_no_rx", "model_df_no_rx_tc"):
        if _name in dir():
            globals()[_name] = (
                globals()[_name]
                .drop(columns=["RACETHX"], errors="ignore")
                .merge(_eth, on="DUPERSID", how="left")
            )
    print(f"RACETHX non-null: {model_df['RACETHX'].notna().sum():,} / {len(model_df):,}")
    print(model_df["RACETHX"].value_counts(dropna=False).sort_index().to_string())
else:
    print("WARN: RACETHX not in demo_df — pull it from h251 first.")


# --- 2. medication_freq_bin ---
def _freq_bin(f):
    if pd.isna(f): return pd.NA
    if f <= 1.25: return "1x_daily"
    if f <= 2.25: return "2x_daily"
    if f <= 3.25: return "3x_daily"
    return "4x_plus"

for _name in ("model_df", "model_df_no_rx", "model_df_no_rx_tc"):
    if _name in dir():
        _df = globals()[_name]
        if "medication_freq" in _df.columns:
            _df["medication_freq_bin"] = _df["medication_freq"].apply(_freq_bin)
            globals()[_name] = _df

print("\nmedication_freq_bin distribution (model_df):")
print(model_df["medication_freq_bin"].value_counts(dropna=False).to_string())


In [ ]:
# ============================================================================
# Model 1 — baseline (RXNAME kept)
# ============================================================================
# Same feature set as the RF baseline in cells 91-92: demographics, insurance, cost, ICD dummies, and RXNAME dummies (RX_*). Answers: how far does XGBoost get on the same inputs RF used? n_drugs and n_conditions are DROPPED here to match RF baseline; Model 2 adds them back.
#
# Self-contained: builds its own X/y, runs HalvingGridSearchCV on the full
# XGB_PARAM_GRID from cell 101, prints holdout metrics, then Gini (gain) and
# SHAP for interpretability. Uses the shared plot_cm_with_recall helper from
# cell 89 for the confusion matrix (recall % per row + accuracy in title).
# All plots use separate figures so they do NOT overlap.

RANDOM_STATE_M1 = 42
TEST_SIZE_M1   = 0.2
CV_FOLDS_M1    = 5
TOP_N_M1       = 15
SHAP_SAMPLE_M1 = 500

# --- build feature matrix ---
_LEAK_M1 = ["DUPERSID", "is_adherent", "meps_adherence_ratio",
              "total_valid_days", "total_days_supply", "drug_start_days"]

def _build_m1(df):
    y = df["is_adherent"].astype(int)
    X = df.drop(columns=[c for c in _LEAK_M1 if c in df.columns], errors="ignore").copy()
    X = X.drop(columns=["n_drugs", "n_conditions"], errors="ignore")
    # (no extra categoricals for this model)
    X = X.select_dtypes(include=[np.number, "bool"]).astype(np.float32)
    return X.fillna(-999.0), y

X_m1, y_m1 = _build_m1(model_df)
print(f"Model 1 (baseline (RXNAME kept)): {X_m1.shape[1]} features, {len(X_m1):,} rows")

X_tr_m1, X_te_m1, y_tr_m1, y_te_m1 = train_test_split(
    X_m1, y_m1,
    test_size=TEST_SIZE_M1,
    random_state=RANDOM_STATE_M1,
    stratify=y_m1,
)

# --- HalvingGridSearchCV on the full XGB_PARAM_GRID ---
cv_m1 = StratifiedKFold(n_splits=CV_FOLDS_M1, shuffle=True, random_state=RANDOM_STATE_M1)
search_m1 = HalvingGridSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", n_jobs=1,
        random_state=RANDOM_STATE_M1, verbosity=0,
    ),
    param_grid=XGB_PARAM_GRID,
    scoring="roc_auc",
    cv=cv_m1,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE_M1,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
_t0 = time.perf_counter()
search_m1.fit(X_tr_m1, y_tr_m1)
model_m1 = search_m1.best_estimator_
print(f"\nHalving done in {(time.perf_counter() - _t0)/60:.1f} min | "
      f"iterations={search_m1.n_iterations_}")
print(f"Best CV AUC: {search_m1.best_score_:.4f}")
print(f"Best params: {search_m1.best_params_}")

# --- holdout metrics ---
pred_m1  = model_m1.predict(X_te_m1)
proba_m1 = model_m1.predict_proba(X_te_m1)[:, 1]
print(f"\nTest AUC:       {roc_auc_score(y_te_m1, proba_m1):.4f}")
print(f"Test F1:        {f1_score(y_te_m1, pred_m1):.4f}")
print(f"Test precision: {precision_score(y_te_m1, pred_m1):.4f}")
print(f"Test recall:    {recall_score(y_te_m1, pred_m1):.4f}")
print("\nClassification report:")
print(classification_report(y_te_m1, pred_m1, digits=3))

# --- Plot 1: confusion matrix (separate figure) ---
fig1, ax1 = plt.subplots(figsize=(6, 5))
plot_cm_with_recall(
    confusion_matrix(y_te_m1, pred_m1),
    ax1,
    "Model 1 (baseline) — confusion matrix",
)
plt.tight_layout()
plt.show()

# --- Plot 2: ROC curve (separate figure) ---
fig2, ax2 = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_te_m1, proba_m1, ax=ax2)
ax2.set_title("Model 1 (baseline) — ROC curve")
plt.tight_layout()
plt.show()

# --- Plot 3: Gini (gain) top-N (separate figure) ---
gain_m1 = (
    pd.Series(model_m1.feature_importances_, index=X_tr_m1.columns)
    .sort_values(ascending=False)
)
print(f"\nTop {TOP_N_M1} by gain:")
display(gain_m1.head(TOP_N_M1).to_frame("gain"))

fig3, ax3 = plt.subplots(figsize=(9, 6))
gain_m1.head(TOP_N_M1).iloc[::-1].plot(kind="barh", ax=ax3, color="steelblue")
ax3.set_xlabel("gain")
ax3.set_title(f"Model 1 (baseline) — top {TOP_N_M1} features by gain")
plt.tight_layout()
plt.show()

# --- Plot 4: SHAP beeswarm (separate figure) ---
X_shap_m1 = X_tr_m1.sample(
    n=min(SHAP_SAMPLE_M1, len(X_tr_m1)),
    random_state=RANDOM_STATE_M1,
)
explainer_m1 = shap.TreeExplainer(model_m1)
shap_values_m1 = explainer_m1.shap_values(X_shap_m1)
mean_abs_shap_m1 = (
    pd.Series(np.abs(shap_values_m1).mean(axis=0), index=X_shap_m1.columns)
    .sort_values(ascending=False)
)
print(f"\nTop {TOP_N_M1} by mean |SHAP| (n={len(X_shap_m1)}):")
display(mean_abs_shap_m1.head(TOP_N_M1).to_frame("mean_abs_shap"))

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_m1,
    X_shap_m1,
    max_display=TOP_N_M1,
    show=False,
)
plt.title(f"Model 1 (baseline) — SHAP beeswarm")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# Model 2 — + n_drugs + n_conditions
# ============================================================================
# Model 1 + polypharmacy count (n_drugs) + comorbidity count (n_conditions). Answers: do those two count features add signal on top of the RXNAME dummies?
#
# Self-contained: builds its own X/y, runs HalvingGridSearchCV on the full
# XGB_PARAM_GRID from cell 101, prints holdout metrics, then Gini (gain) and
# SHAP for interpretability. Uses the shared plot_cm_with_recall helper from
# cell 89 for the confusion matrix (recall % per row + accuracy in title).
# All plots use separate figures so they do NOT overlap.

RANDOM_STATE_M2 = 42
TEST_SIZE_M2   = 0.2
CV_FOLDS_M2    = 5
TOP_N_M2       = 15
SHAP_SAMPLE_M2 = 500

# --- build feature matrix ---
_LEAK_M2 = ["DUPERSID", "is_adherent", "meps_adherence_ratio",
              "total_valid_days", "total_days_supply", "drug_start_days"]

def _build_m2(df):
    y = df["is_adherent"].astype(int)
    X = df.drop(columns=[c for c in _LEAK_M2 if c in df.columns], errors="ignore").copy()
    # n_drugs / n_conditions kept as features (polypharmacy + comorbidity signal)
    # (no extra categoricals for this model)
    X = X.select_dtypes(include=[np.number, "bool"]).astype(np.float32)
    return X.fillna(-999.0), y

X_m2, y_m2 = _build_m2(model_df)
print(f"Model 2 (+ n_drugs + n_conditions): {X_m2.shape[1]} features, {len(X_m2):,} rows")

X_tr_m2, X_te_m2, y_tr_m2, y_te_m2 = train_test_split(
    X_m2, y_m2,
    test_size=TEST_SIZE_M2,
    random_state=RANDOM_STATE_M2,
    stratify=y_m2,
)

# --- HalvingGridSearchCV on the full XGB_PARAM_GRID ---
cv_m2 = StratifiedKFold(n_splits=CV_FOLDS_M2, shuffle=True, random_state=RANDOM_STATE_M2)
search_m2 = HalvingGridSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", n_jobs=1,
        random_state=RANDOM_STATE_M2, verbosity=0,
    ),
    param_grid=XGB_PARAM_GRID,
    scoring="roc_auc",
    cv=cv_m2,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE_M2,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
_t0 = time.perf_counter()
search_m2.fit(X_tr_m2, y_tr_m2)
model_m2 = search_m2.best_estimator_
print(f"\nHalving done in {(time.perf_counter() - _t0)/60:.1f} min | "
      f"iterations={search_m2.n_iterations_}")
print(f"Best CV AUC: {search_m2.best_score_:.4f}")
print(f"Best params: {search_m2.best_params_}")

# --- holdout metrics ---
pred_m2  = model_m2.predict(X_te_m2)
proba_m2 = model_m2.predict_proba(X_te_m2)[:, 1]
print(f"\nTest AUC:       {roc_auc_score(y_te_m2, proba_m2):.4f}")
print(f"Test F1:        {f1_score(y_te_m2, pred_m2):.4f}")
print(f"Test precision: {precision_score(y_te_m2, pred_m2):.4f}")
print(f"Test recall:    {recall_score(y_te_m2, pred_m2):.4f}")
print("\nClassification report:")
print(classification_report(y_te_m2, pred_m2, digits=3))

# --- Plot 1: confusion matrix (separate figure) ---
fig1, ax1 = plt.subplots(figsize=(6, 5))
plot_cm_with_recall(
    confusion_matrix(y_te_m2, pred_m2),
    ax1,
    "Model 2 (+n_drugs, +n_conditions) — confusion matrix",
)
plt.tight_layout()
plt.show()

# --- Plot 2: ROC curve (separate figure) ---
fig2, ax2 = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_te_m2, proba_m2, ax=ax2)
ax2.set_title("Model 2 (+n_drugs, +n_conditions) — ROC curve")
plt.tight_layout()
plt.show()

# --- Plot 3: Gini (gain) top-N (separate figure) ---
gain_m2 = (
    pd.Series(model_m2.feature_importances_, index=X_tr_m2.columns)
    .sort_values(ascending=False)
)
print(f"\nTop {TOP_N_M2} by gain:")
display(gain_m2.head(TOP_N_M2).to_frame("gain"))

fig3, ax3 = plt.subplots(figsize=(9, 6))
gain_m2.head(TOP_N_M2).iloc[::-1].plot(kind="barh", ax=ax3, color="steelblue")
ax3.set_xlabel("gain")
ax3.set_title(f"Model 2 (+n_drugs, +n_conditions) — top {TOP_N_M2} features by gain")
plt.tight_layout()
plt.show()

# --- Plot 4: SHAP beeswarm (separate figure) ---
X_shap_m2 = X_tr_m2.sample(
    n=min(SHAP_SAMPLE_M2, len(X_tr_m2)),
    random_state=RANDOM_STATE_M2,
)
explainer_m2 = shap.TreeExplainer(model_m2)
shap_values_m2 = explainer_m2.shap_values(X_shap_m2)
mean_abs_shap_m2 = (
    pd.Series(np.abs(shap_values_m2).mean(axis=0), index=X_shap_m2.columns)
    .sort_values(ascending=False)
)
print(f"\nTop {TOP_N_M2} by mean |SHAP| (n={len(X_shap_m2)}):")
display(mean_abs_shap_m2.head(TOP_N_M2).to_frame("mean_abs_shap"))

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_m2,
    X_shap_m2,
    max_display=TOP_N_M2,
    show=False,
)
plt.title(f"Model 2 (+n_drugs, +n_conditions) — SHAP beeswarm")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# Model 3 — drug class instead of drug name
# ============================================================================
# Model 2 with RXNAME dummies REPLACED by TC1/TC1S1 drug-class dummies (already in model_df_no_rx_tc). Answers: does a coarser drug representation (class not name) generalize better? Keeps n_drugs and n_conditions from Model 2.
#
# Self-contained: builds its own X/y, runs HalvingGridSearchCV on the full
# XGB_PARAM_GRID from cell 101, prints holdout metrics, then Gini (gain) and
# SHAP for interpretability. Uses the shared plot_cm_with_recall helper from
# cell 89 for the confusion matrix (recall % per row + accuracy in title).
# All plots use separate figures so they do NOT overlap.

RANDOM_STATE_M3 = 42
TEST_SIZE_M3   = 0.2
CV_FOLDS_M3    = 5
TOP_N_M3       = 15
SHAP_SAMPLE_M3 = 500

# --- build feature matrix ---
_LEAK_M3 = ["DUPERSID", "is_adherent", "meps_adherence_ratio",
              "total_valid_days", "total_days_supply", "drug_start_days"]

def _build_m3(df):
    y = df["is_adherent"].astype(int)
    X = df.drop(columns=[c for c in _LEAK_M3 if c in df.columns], errors="ignore").copy()
    X = X.drop(columns=[c for c in X.columns if c.startswith("RX_")], errors="ignore")
    # n_drugs / n_conditions kept as features (polypharmacy + comorbidity signal)
    # (no extra categoricals for this model)
    X = X.select_dtypes(include=[np.number, "bool"]).astype(np.float32)
    return X.fillna(-999.0), y

X_m3, y_m3 = _build_m3(model_df_no_rx_tc)
print(f"Model 3 (drug class instead of drug name): {X_m3.shape[1]} features, {len(X_m3):,} rows")

X_tr_m3, X_te_m3, y_tr_m3, y_te_m3 = train_test_split(
    X_m3, y_m3,
    test_size=TEST_SIZE_M3,
    random_state=RANDOM_STATE_M3,
    stratify=y_m3,
)

# --- HalvingGridSearchCV on the full XGB_PARAM_GRID ---
cv_m3 = StratifiedKFold(n_splits=CV_FOLDS_M3, shuffle=True, random_state=RANDOM_STATE_M3)
search_m3 = HalvingGridSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", n_jobs=1,
        random_state=RANDOM_STATE_M3, verbosity=0,
    ),
    param_grid=XGB_PARAM_GRID,
    scoring="roc_auc",
    cv=cv_m3,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE_M3,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
_t0 = time.perf_counter()
search_m3.fit(X_tr_m3, y_tr_m3)
model_m3 = search_m3.best_estimator_
print(f"\nHalving done in {(time.perf_counter() - _t0)/60:.1f} min | "
      f"iterations={search_m3.n_iterations_}")
print(f"Best CV AUC: {search_m3.best_score_:.4f}")
print(f"Best params: {search_m3.best_params_}")

# --- holdout metrics ---
pred_m3  = model_m3.predict(X_te_m3)
proba_m3 = model_m3.predict_proba(X_te_m3)[:, 1]
print(f"\nTest AUC:       {roc_auc_score(y_te_m3, proba_m3):.4f}")
print(f"Test F1:        {f1_score(y_te_m3, pred_m3):.4f}")
print(f"Test precision: {precision_score(y_te_m3, pred_m3):.4f}")
print(f"Test recall:    {recall_score(y_te_m3, pred_m3):.4f}")
print("\nClassification report:")
print(classification_report(y_te_m3, pred_m3, digits=3))

# --- Plot 1: confusion matrix (separate figure) ---
fig1, ax1 = plt.subplots(figsize=(6, 5))
plot_cm_with_recall(
    confusion_matrix(y_te_m3, pred_m3),
    ax1,
    "Model 3 (drug class) — confusion matrix",
)
plt.tight_layout()
plt.show()

# --- Plot 2: ROC curve (separate figure) ---
fig2, ax2 = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_te_m3, proba_m3, ax=ax2)
ax2.set_title("Model 3 (drug class) — ROC curve")
plt.tight_layout()
plt.show()

# --- Plot 3: Gini (gain) top-N (separate figure) ---
gain_m3 = (
    pd.Series(model_m3.feature_importances_, index=X_tr_m3.columns)
    .sort_values(ascending=False)
)
print(f"\nTop {TOP_N_M3} by gain:")
display(gain_m3.head(TOP_N_M3).to_frame("gain"))

fig3, ax3 = plt.subplots(figsize=(9, 6))
gain_m3.head(TOP_N_M3).iloc[::-1].plot(kind="barh", ax=ax3, color="steelblue")
ax3.set_xlabel("gain")
ax3.set_title(f"Model 3 (drug class) — top {TOP_N_M3} features by gain")
plt.tight_layout()
plt.show()

# --- Plot 4: SHAP beeswarm (separate figure) ---
X_shap_m3 = X_tr_m3.sample(
    n=min(SHAP_SAMPLE_M3, len(X_tr_m3)),
    random_state=RANDOM_STATE_M3,
)
explainer_m3 = shap.TreeExplainer(model_m3)
shap_values_m3 = explainer_m3.shap_values(X_shap_m3)
mean_abs_shap_m3 = (
    pd.Series(np.abs(shap_values_m3).mean(axis=0), index=X_shap_m3.columns)
    .sort_values(ascending=False)
)
print(f"\nTop {TOP_N_M3} by mean |SHAP| (n={len(X_shap_m3)}):")
display(mean_abs_shap_m3.head(TOP_N_M3).to_frame("mean_abs_shap"))

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_m3,
    X_shap_m3,
    max_display=TOP_N_M3,
    show=False,
)
plt.title(f"Model 3 (drug class) — SHAP beeswarm")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# Model 4 — paper-enriched
# ============================================================================
# Model 3 + socioeconomic (MARRYXX, REGIONXX, EDUCYR, RACETHX one-hot) + medication_dose + medication_freq + medication_freq_bin (one-hot). Tests the scoping review's 'socioeconomic factors are strong predictors' recommendation and Toy et al.'s dosing-frequency finding.
#
# Self-contained: builds its own X/y, runs HalvingGridSearchCV on the full
# XGB_PARAM_GRID from cell 101, prints holdout metrics, then Gini (gain) and
# SHAP for interpretability. Uses the shared plot_cm_with_recall helper from
# cell 89 for the confusion matrix (recall % per row + accuracy in title).
# All plots use separate figures so they do NOT overlap.

RANDOM_STATE_M4 = 42
TEST_SIZE_M4   = 0.2
CV_FOLDS_M4    = 5
TOP_N_M4       = 15
SHAP_SAMPLE_M4 = 500

# --- build feature matrix ---
_LEAK_M4 = ["DUPERSID", "is_adherent", "meps_adherence_ratio",
              "total_valid_days", "total_days_supply", "drug_start_days"]

def _build_m4(df):
    y = df["is_adherent"].astype(int)
    X = df.drop(columns=[c for c in _LEAK_M4 if c in df.columns], errors="ignore").copy()
    X = X.drop(columns=[c for c in X.columns if c.startswith("RX_")], errors="ignore")
    # n_drugs / n_conditions kept as features (polypharmacy + comorbidity signal)
    for _c in ["MARRYXX", "REGIONXX", "RACETHX", "medication_freq_bin"]:
        if _c in X.columns:
            X = pd.get_dummies(X, columns=[_c], prefix=_c, dummy_na=True)
    X = X.select_dtypes(include=[np.number, "bool"]).astype(np.float32)
    return X.fillna(-999.0), y

X_m4, y_m4 = _build_m4(model_df_no_rx_tc)
print(f"Model 4 (paper-enriched): {X_m4.shape[1]} features, {len(X_m4):,} rows")

X_tr_m4, X_te_m4, y_tr_m4, y_te_m4 = train_test_split(
    X_m4, y_m4,
    test_size=TEST_SIZE_M4,
    random_state=RANDOM_STATE_M4,
    stratify=y_m4,
)

# --- HalvingGridSearchCV on the full XGB_PARAM_GRID ---
cv_m4 = StratifiedKFold(n_splits=CV_FOLDS_M4, shuffle=True, random_state=RANDOM_STATE_M4)
search_m4 = HalvingGridSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", n_jobs=1,
        random_state=RANDOM_STATE_M4, verbosity=0,
    ),
    param_grid=XGB_PARAM_GRID,
    scoring="roc_auc",
    cv=cv_m4,
    factor=3,
    resource="n_samples",
    random_state=RANDOM_STATE_M4,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
_t0 = time.perf_counter()
search_m4.fit(X_tr_m4, y_tr_m4)
model_m4 = search_m4.best_estimator_
print(f"\nHalving done in {(time.perf_counter() - _t0)/60:.1f} min | "
      f"iterations={search_m4.n_iterations_}")
print(f"Best CV AUC: {search_m4.best_score_:.4f}")
print(f"Best params: {search_m4.best_params_}")

# --- holdout metrics ---
pred_m4  = model_m4.predict(X_te_m4)
proba_m4 = model_m4.predict_proba(X_te_m4)[:, 1]
print(f"\nTest AUC:       {roc_auc_score(y_te_m4, proba_m4):.4f}")
print(f"Test F1:        {f1_score(y_te_m4, pred_m4):.4f}")
print(f"Test precision: {precision_score(y_te_m4, pred_m4):.4f}")
print(f"Test recall:    {recall_score(y_te_m4, pred_m4):.4f}")
print("\nClassification report:")
print(classification_report(y_te_m4, pred_m4, digits=3))

# --- Plot 1: confusion matrix (separate figure) ---
fig1, ax1 = plt.subplots(figsize=(6, 5))
plot_cm_with_recall(
    confusion_matrix(y_te_m4, pred_m4),
    ax1,
    "Model 4 (paper enriched) — confusion matrix",
)
plt.tight_layout()
plt.show()

# --- Plot 2: ROC curve (separate figure) ---
fig2, ax2 = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_te_m4, proba_m4, ax=ax2)
ax2.set_title("Model 4 (paper enriched) — ROC curve")
plt.tight_layout()
plt.show()

# --- Plot 3: Gini (gain) top-N (separate figure) ---
gain_m4 = (
    pd.Series(model_m4.feature_importances_, index=X_tr_m4.columns)
    .sort_values(ascending=False)
)
print(f"\nTop {TOP_N_M4} by gain:")
display(gain_m4.head(TOP_N_M4).to_frame("gain"))

fig3, ax3 = plt.subplots(figsize=(9, 6))
gain_m4.head(TOP_N_M4).iloc[::-1].plot(kind="barh", ax=ax3, color="steelblue")
ax3.set_xlabel("gain")
ax3.set_title(f"Model 4 (paper enriched) — top {TOP_N_M4} features by gain")
plt.tight_layout()
plt.show()

# --- Plot 4: SHAP beeswarm (separate figure) ---
X_shap_m4 = X_tr_m4.sample(
    n=min(SHAP_SAMPLE_M4, len(X_tr_m4)),
    random_state=RANDOM_STATE_M4,
)
explainer_m4 = shap.TreeExplainer(model_m4)
shap_values_m4 = explainer_m4.shap_values(X_shap_m4)
mean_abs_shap_m4 = (
    pd.Series(np.abs(shap_values_m4).mean(axis=0), index=X_shap_m4.columns)
    .sort_values(ascending=False)
)
print(f"\nTop {TOP_N_M4} by mean |SHAP| (n={len(X_shap_m4)}):")
display(mean_abs_shap_m4.head(TOP_N_M4).to_frame("mean_abs_shap"))

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_m4,
    X_shap_m4,
    max_display=TOP_N_M4,
    show=False,
)
plt.title(f"Model 4 (paper enriched) — SHAP beeswarm")
plt.tight_layout()
plt.show()
